# 🌊 WRAI-X (0.6B) — 1-CLICK ZERO-CONTAMINATION TRANSPLANT & RAPID TRAINER

Notebook ini **100% mandiri (Standalone)**. Anda tidak perlu upload file python lain.

### ⚡ Fitur Utama:
1. **Auto-Install Dependensi**: Otomatis menginstall 	ransformers, datasets, ccelerate.
2. **Auto-Mount Google Drive**: Hasil binary akan langsung tersimpan di Google Drive (/MyDrive/WRAI_X_06B/).
3. **Surgical 1-to-1 Transplant**: Bobot Qwen 0.6B disalin utuh tanpa pemotongan vocab.
4. **Frozen Knowledge (Anti-Kontaminasi)**: 70% otak Qwen (FFN 264M + Vocab 155M) DIGEMBOK TOTAL.
5. **Fast Adapter Training**: Hanya melatih Retention Decays, Haar DWT, HDC, dan Thinking Gate (~7-10 menit di T4 GPU).
6. **Auto-Quantize INT8 C Native**: Menghasilkan wrai_x_06b_int8.bin (~540 MB) & wrai_x_vocab.bin siap jalan di laptop AMD A8!

---
**Cukup tekan Ctrl + F9 atau klik Runtime -> Run all!**

In [ ]:
# 1. Bersihkan memory CUDA & Konfigurasi Allocator
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import gc
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

!nvidia-smi
# 2. Install Dependensi
!pip install -q transformers datasets accelerate


In [ ]:
#!/usr/bin/env python3
"""
=============================================================================
 🌊 WRAI-X (0.6B) — SURGICAL ZERO-CONTAMINATION TRANSPLANT & RAPID ADAPTER
=============================================================================
 Arsitektur:
  - 100% Tanpa KV-Cache (O(1) Recurrent Ping-Pong Buffer)
  - 28 Layer Dual-State (Memory Mt + Reasoning Rt)
  - 4-Level Haar Multiresolution Spectral Filter (D=1024 power-of-2)
  - HDC (Hyperdimensional Computing) Associative Scratchpad
  - Dimensi: Hidden=1024, FFN=3072, 16 Heads, Head Dim=128, Layers=28, Vocab=151936

 Prinsip Anti-Kontaminasi:
  1. 100% Bedah Cangkok 1-to-1: FFN (264M params), Embeddings (155M params),
     dan LayerNorms dicopy BIT-LEVEL IDENTIK dari Qwen3-0.6B.
  2. Frozen Core Knowledge: Seluruh bobot FFN dan Embedding digembok (requires_grad = False).
     Dataset TIDAK AKAN PERNAH BISA menimpa pengetahuan bahasa atau fakta Qwen!
  3. Pattern-Only Training: Dataset HANYA melatih parameter pola WRAI-X:
     - Decay parameter gamma (transisi Attention -> Retention)
     - Haar Multiresolution Gating & Gains
     - Adaptive Thinking Gate (gk)
     - HDC Associative Scratchpad
=============================================================================
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import math
import time
import json
import random
import gc
import struct
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import subprocess
for pkg in ["transformers", "datasets", "accelerate"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"[*] Menginstall dependensi {pkg} secara otomatis...", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from transformers.optimization import Adafactor
from datasets import load_dataset

# -----------------------------------------------------------------------------
# 1. Konfigurasi Arsitektur WRAI-X (0.6B)
# -----------------------------------------------------------------------------
SOURCE_MODEL_NAME = "Qwen/Qwen3-0.6B"

HIDDEN_DIM = 1024          # D = 1024 (2^10 murni)
FFN_DIM = 3072             # SwiGLU Intermediate Size
NUM_LAYERS = 28            # 28 Layers (1-to-1 dengan Qwen 0.6B)
NUM_HEADS = 16             # 16 Retention Heads
HEAD_DIM = 128             # 16 x 128 = 2048
WAVELET_LEVELS = 4         # 4-Level Haar DWT
VOCAB_SIZE = 151936        # Qwen 3 Vocab Size

MAX_SEQ_LEN = 128          # 128 Token agar seluruh jawaban CoT & Indonesia selesai bersama <|im_end|>
BATCH_SIZE = 2             # Micro-batching untuk keamanan total VRAM GPU T4
GRAD_ACCUM_STEPS = 5       # 5 micro-batches per step (10 sampel total)
LEARNING_RATE = 5e-4       # Stable adaptation rate for retention routers
WEIGHT_DECAY = 0.01

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------------
# 2. Modul Arsitektur WRAI-X
# -----------------------------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * norm).to(x.dtype) * self.weight

class SwiGLUFFN(nn.Module):
    def __init__(self, hidden_dim=1024, ffn_dim=3072):
        super().__init__()
        self.w_gate = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.w_up   = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.w_down = nn.Linear(ffn_dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class HaarMultiresolution1D(nn.Module):
    """
    Fase 3: Haar 1D Multiresolution Hierarchy pada Hidden Dimension D=1024.
    Dekomposisi frekuensi:
      - High Frequency (512) : Permukaan sintaks -> residual bypass
      - Mid Frequency  (384) : Komposisi logika -> Reasoning State
      - Low Frequency  (64)  : Fakta inti & konteks persisten -> Reasoning State
    """
    def __init__(self, dim=1024, levels=4):
        super().__init__()
        self.dim = dim
        self.levels = levels
        self.sqrt2 = math.sqrt(2.0)
        self.high_gain = nn.Parameter(torch.ones(1))
        self.mid_gain  = nn.Parameter(torch.ones(1))
        self.low_gain  = nn.Parameter(torch.ones(1))
        self.gate_w = nn.Parameter(torch.zeros(dim))
        self.gate_b = nn.Parameter(torch.full((dim,), -5.0)) # Transparent at Step 0

    def forward(self, x):
        # x: (N, D)
        N, D = x.shape
        approx = x
        details = []

        for lvl in range(self.levels):
            even = approx[:, 0::2]
            odd  = approx[:, 1::2]
            a = (even + odd) / self.sqrt2
            d = (even - odd) / self.sqrt2
            details.append(d)
            approx = a

        low_band = approx * self.low_gain
        mid_band = torch.cat([details[1], details[2], details[3]], dim=-1) * self.mid_gain
        high_band = details[0] * self.high_gain

        rec_approx = low_band
        for lvl in reversed(range(self.levels)):
            d = details[lvl]
            if lvl == 0:
                d = high_band
            even_rec = (rec_approx + d) / self.sqrt2
            odd_rec  = (rec_approx - d) / self.sqrt2
            merged = torch.empty(N, even_rec.size(1) * 2, device=x.device, dtype=x.dtype)
            merged[:, 0::2] = even_rec
            merged[:, 1::2] = odd_rec
            rec_approx = merged

        gate = torch.sigmoid(x * self.gate_w + self.gate_b)
        x_filtered = x + (rec_approx * gate)
        return x_filtered, low_band, mid_band

class HDCAssociativeScratchpad(nn.Module):
    """Fase 4: Hyperdimensional Computing (HDC) Associative Scratchpad"""
    def __init__(self, dim=1024):
        super().__init__()
        self.dim = dim
        self.proj_key = nn.Linear(dim, dim, bias=False)
        self.proj_val = nn.Linear(dim, dim, bias=False)
        self.gate_hdc = nn.Linear(dim * 2, dim, bias=True)
        # Inisialisasi transparan: bias negatif agar gate tertutup di step 0
        nn.init.constant_(self.gate_hdc.bias, -5.0)

    def forward(self, r_t, scratchpad_state=None):
        k = torch.tanh(self.proj_key(r_t))
        v = torch.tanh(self.proj_val(r_t))
        bound = k * v
        if scratchpad_state is None:
            new_scratchpad = bound
        else:
            new_scratchpad = scratchpad_state * 0.95 + bound
        resonance = new_scratchpad * k
        g = torch.sigmoid(self.gate_hdc(torch.cat([r_t, resonance], dim=-1)))
        out = r_t + (resonance * g)
        return out, new_scratchpad

def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

class WRAIRotaryEmbedding(nn.Module):
    def __init__(self, dim, base=1000000.0):
        super().__init__()
        self.dim = dim
        self.register_buffer("inv_freq", 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim)), persistent=False)

    def get_cos_sin(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device).float()
        freqs = torch.outer(t, self.inv_freq.to(device))
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def apply_parallel(self, x, cos, sin):
        return (x * cos.view(1, 1, cos.size(0), cos.size(1))) + (rotate_half(x) * sin.view(1, 1, sin.size(0), sin.size(1)))

    def apply_step(self, x, step_pos, device, dtype):
        t = torch.tensor([step_pos], device=device).float()
        freqs = torch.outer(t, self.inv_freq.to(device))
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos().to(dtype).view(1, 1, self.dim)
        sin = emb.sin().to(dtype).view(1, 1, self.dim)
        return (x * cos) + (rotate_half(x) * sin)

class LoRALinear(nn.Module):
    """LoRA Precision Adapter Rank-4: Zero-init delta bridge to avoid catastrophic drift"""
    def __init__(self, base_linear, rank=4, alpha=8.0):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False
        in_dim = base_linear.in_features
        out_dim = base_linear.out_features
        self.rank = rank
        self.scale = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(rank, in_dim) * (1.0 / math.sqrt(in_dim)))
        self.lora_B = nn.Parameter(torch.zeros(out_dim, rank))

    def forward(self, x):
        base_out = self.base(x)
        lora_out = F.linear(x, self.lora_B @ self.lora_A) * self.scale
        return base_out + lora_out

    def merge_and_restore(self):
        delta = (self.lora_B @ self.lora_A) * self.scale
        self.base.weight.data.add_(delta)
        return self.base

class WRAIXDualStateBlock(nn.Module):
    def __init__(self, hidden_dim=1024, ffn_dim=3072, num_heads=16, head_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.scale = 1.0 / math.sqrt(head_dim)

        self.rope = WRAIRotaryEmbedding(dim=head_dim)

        # 1. Memory State (Mt) Projections (Dicangkok 1:1 dari Attention Qwen)
        self.rms_ret = RMSNorm(hidden_dim)
        self.w_q = nn.Linear(hidden_dim, num_heads * head_dim, bias=False)
        self.w_k = nn.Linear(hidden_dim, num_heads * head_dim, bias=False)
        self.w_v = nn.Linear(hidden_dim, num_heads * head_dim, bias=False)
        self.w_out = nn.Linear(num_heads * head_dim, hidden_dim, bias=False)
        self.gn_m = nn.GroupNorm(num_groups=num_heads, num_channels=num_heads * head_dim, affine=True)

        # 2. Reasoning State (Rt) Projections (Adapter Pola Baru)
        self.w_qr = nn.Linear(hidden_dim, num_heads * head_dim, bias=False)
        self.w_kr = nn.Linear(hidden_dim, num_heads * head_dim, bias=False)
        self.w_vr = nn.Linear(hidden_dim, num_heads * head_dim, bias=False)
        self.w_out_r = nn.Linear(num_heads * head_dim, hidden_dim, bias=False)
        self.gn_r = nn.GroupNorm(num_groups=num_heads, num_channels=num_heads * head_dim, affine=True)

        # Decays (gamma) Memory & Reasoning
        # Rentang peluruhan seimbang: dari half-life 2.7 token (subword) hingga 102.5 token (konteks)
        init_gammas = 1.0 - torch.exp(-torch.linspace(1.5, 5.0, num_heads))
        self.decay_m = nn.Parameter(torch.logit(init_gammas))
        self.decay_r = nn.Parameter(torch.logit(init_gammas * 0.98))

        # 3. Haar Spectral Bridge
        self.haar_bridge = HaarMultiresolution1D(dim=hidden_dim, levels=WAVELET_LEVELS)

        # 4. Adaptive Thinking Gate: gk = alpha * sigma(Wg [M, R]) (Transparan di awal)
        self.think_gate = nn.Linear(hidden_dim * 2, hidden_dim, bias=True)
        nn.init.constant_(self.think_gate.bias, -5.0)
        nn.init.zeros_(self.think_gate.weight)
        self.alpha = nn.Parameter(torch.zeros(1))

        # 5. HDC Scratchpad
        self.hdc = HDCAssociativeScratchpad(dim=hidden_dim)

        # 6. SwiGLU FFN (Dicangkok 1:1 dari Qwen)
        self.rms_ffn = RMSNorm(hidden_dim)
        self.ffn = SwiGLUFFN(hidden_dim, ffn_dim)

    def forward_parallel(self, x):
        # x: (B, T, D) - O(1) Sequential Operations Training
        B, T, D = x.shape
        H, HD = self.num_heads, self.head_dim

        x_norm = self.rms_ret(x)

        # 1. RoPE + Multi-Scale Retention (Mt) with Per-Head RMSNorm
        q = self.w_q(x_norm).view(B, T, H, HD).transpose(1, 2)
        k = self.w_k(x_norm).view(B, T, H, HD).transpose(1, 2)
        v = self.w_v(x_norm).view(B, T, H, HD).transpose(1, 2)

        cos, sin = self.rope.get_cos_sin(T, x.device, x.dtype)
        q_rope = self.rope.apply_parallel(q, cos, sin)
        k_rope = self.rope.apply_parallel(k, cos, sin)

        gamma_m = torch.sigmoid(self.decay_m).view(1, H, 1, 1)
        i_idx = torch.arange(T, device=x.device).view(T, 1)
        j_idx = torch.arange(T, device=x.device).view(1, T)
        dist = (i_idx - j_idx).clamp(min=0).view(1, 1, T, T).to(x.dtype)
        causal = (i_idx >= j_idx).view(1, 1, T, T).to(x.dtype)
        decay_m = (torch.pow(gamma_m, dist) * causal).to(q.dtype)

        # Retention dot-product: (scale * Q K^T * D) V
        scores_m = torch.matmul(q_rope * self.scale, k_rope.transpose(-1, -2)) * decay_m
        o_head_m = torch.matmul(scores_m, v).transpose(1, 2).contiguous() # (B, T, H, HD)
        o_m = self.w_out(self.gn_m(o_head_m.view(B * T, H * HD)).view(B, T, H * HD))

        # 2. Haar Multiresolution Bridge
        o_m_flat = o_m.view(B * T, D)
        o_m_filtered_flat, low_band, mid_band = self.haar_bridge(o_m_flat)
        o_m_filtered = o_m_filtered_flat.view(B, T, D)

        # 3. RoPE + Multi-Scale Retention (Rt) with GroupNorm (Zero-Mean Centering)
        qr = self.w_qr(o_m_filtered).view(B, T, H, HD).transpose(1, 2)
        kr = self.w_kr(o_m_filtered).view(B, T, H, HD).transpose(1, 2)
        vr = self.w_vr(o_m_filtered).view(B, T, H, HD).transpose(1, 2)

        qr_rope = self.rope.apply_parallel(qr, cos, sin)
        kr_rope = self.rope.apply_parallel(kr, cos, sin)

        gamma_r = torch.sigmoid(self.decay_r).view(1, H, 1, 1)
        decay_r = (torch.pow(gamma_r, dist) * causal).to(qr.dtype)

        scores_r = torch.matmul(qr_rope * self.scale, kr_rope.transpose(-1, -2)) * decay_r
        o_head_r = torch.matmul(scores_r, vr).transpose(1, 2).contiguous() # (B, T, H, HD)
        o_r = self.w_out_r(self.gn_r(o_head_r.view(B * T, H * HD)).view(B, T, H * HD))

        # 4. HDC Associative Scratchpad Parallel
        k_hdc = torch.tanh(self.hdc.proj_key(o_r))
        v_hdc = torch.tanh(self.hdc.proj_val(o_r))
        bound = k_hdc * v_hdc
        decay_hdc = ((0.95 ** dist.view(1, T, T)) * causal.view(1, T, T)).to(bound.dtype)
        scratchpad = torch.matmul(decay_hdc, bound)
        res = scratchpad * k_hdc
        g_hdc = torch.sigmoid(self.hdc.gate_hdc(torch.cat([o_r, res], dim=-1)))
        o_r_hdc = o_r + (res * g_hdc)

        # 5. Adaptive Thinking Gate dengan Zero-Init Residual Alpha
        gate_think = torch.sigmoid(self.think_gate(torch.cat([o_m_filtered, o_r_hdc], dim=-1)))
        ret_fused = o_m_filtered + self.alpha * (o_r_hdc * gate_think)

        x = x + ret_fused

        # 6. SwiGLU FFN Sub-layer
        x = x + self.ffn(self.rms_ffn(x))
        return x

    def forward_step(self, x, step_pos=0, state_m=None, state_r=None, state_hdc=None):
        # x: (B, D) - Recurrent Step O(1) Inference with zero KV cache
        B, D = x.shape
        H, HD = self.num_heads, self.head_dim

        x_norm = self.rms_ret(x)

        # 1. RoPE + Memory State (Mt) with Per-Head RMSNorm
        q = self.w_q(x_norm).view(B, H, HD)
        k = self.w_k(x_norm).view(B, H, HD)
        v = self.w_v(x_norm).view(B, H, HD)

        q_rope = self.rope.apply_step(q, step_pos, x.device, x.dtype)
        k_rope = self.rope.apply_step(k, step_pos, x.device, x.dtype)

        gamma_m = torch.sigmoid(self.decay_m).view(1, H, 1, 1)
        if state_m is None:
            state_m = torch.zeros(B, H, HD, HD, device=x.device, dtype=x.dtype)

        # S_m = gamma_m * S_m + (scale * K^T) * V
        state_m = state_m * gamma_m + torch.einsum('bhr,bhc->bhrc', k_rope * self.scale, v)
        # O_m = Q * S_m
        o_head_m = torch.einsum('bhr,bhrc->bhc', q_rope, state_m).reshape(B, H * HD)
        o_m = self.w_out(self.gn_m(o_head_m))

        # 2. Haar Multiresolution Bridge
        o_m_filtered, low_band, mid_band = self.haar_bridge(o_m)

        # 3. RoPE + Reasoning State (Rt) with GroupNorm (Zero-Mean Centering)
        qr = self.w_qr(o_m_filtered).view(B, H, HD)
        kr = self.w_kr(o_m_filtered).view(B, H, HD)
        vr = self.w_vr(o_m_filtered).view(B, H, HD)

        qr_rope = self.rope.apply_step(qr, step_pos, x.device, x.dtype)
        kr_rope = self.rope.apply_step(kr, step_pos, x.device, x.dtype)

        gamma_r = torch.sigmoid(self.decay_r).view(1, H, 1, 1)
        if state_r is None:
            state_r = torch.zeros(B, H, HD, HD, device=x.device, dtype=x.dtype)

        state_r = state_r * gamma_r + torch.einsum('bhr,bhc->bhrc', kr_rope * self.scale, vr)
        o_head_r = torch.einsum('bhr,bhrc->bhc', qr_rope, state_r).reshape(B, H * HD)
        o_r = self.w_out_r(self.gn_r(o_head_r))

        # 4. HDC Associative Scratchpad
        o_r_hdc, state_hdc = self.hdc(o_r, state_hdc)

        # 5. Adaptive Thinking Gate dengan Zero-Init Residual Alpha
        gate_think = torch.sigmoid(self.think_gate(torch.cat([o_m_filtered, o_r_hdc], dim=-1)))
        ret_fused = o_m_filtered + self.alpha * (o_r_hdc * gate_think)

        x = x + ret_fused

        # 6. SwiGLU FFN
        x = x + self.ffn(self.rms_ffn(x))
        return x, state_m, state_r, state_hdc

class WRAIX06BModel(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, num_layers=NUM_LAYERS, hidden_dim=HIDDEN_DIM, ffn_dim=FFN_DIM):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.layers = nn.ModuleList([
            WRAIXDualStateBlock(hidden_dim=hidden_dim, ffn_dim=ffn_dim)
            for _ in range(num_layers)
        ])
        self.ln_final = RMSNorm(hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, vocab_size, bias=False)
        self.output_proj.weight = self.embed.weight

    def forward_parallel(self, input_ids):
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer.forward_parallel(x)
        x_norm = self.ln_final(x)
        return self.output_proj(x_norm)

    def forward_step(self, token_id, states=None):
        x = self.embed(token_id)
        if states is None:
            states = [None] * self.num_layers

        new_states = []
        for l in range(self.num_layers):
            layer_state = states[l] if states[l] is not None else (None, None, None, 0)
            sm, sr, shdc, pos = layer_state
            x, sm, sr, shdc = self.layers[l].forward_step(
                x, step_pos=pos, state_m=sm, state_r=sr, state_hdc=shdc
            )
            new_states.append((sm, sr, shdc, pos + 1))

        x_norm = self.ln_final(x)
        logits = self.output_proj(x_norm)
        return logits, new_states

# -----------------------------------------------------------------------------
# 3. Mesin Cangkok Bedah 1-to-1 dari Qwen 0.6B ke WRAI-X
# -----------------------------------------------------------------------------

def surgical_transplant_qwen_to_wrai_x(wrai_model, source_model_name=SOURCE_MODEL_NAME, return_teacher=False):
    print("=" * 70)
    print(f"[*] MEMULAI CANGKOK BEDAH 1-TO-1 DARI {source_model_name}...")
    print("=" * 70)
    t0 = time.time()

    qwen = AutoModelForCausalLM.from_pretrained(
        source_model_name,
        device_map="cpu",
        trust_remote_code=True
    )
    qwen_sd = qwen.state_dict()

    print("[1/5] Mencangkok Embeddings Utuh (151,936 Token x 1024 Dim)...")
    qwen_embed = qwen_sd["model.embed_tokens.weight"]
    v_copy = min(wrai_model.vocab_size, qwen_embed.size(0))
    wrai_model.embed.weight.data[:v_copy].copy_(qwen_embed[:v_copy])

    print(f"[2/5] Mencangkok 28 Layer SwiGLU FFN & RMSNorms...")
    for l in range(NUM_LAYERS):
        # FFN: Gate, Up, Down (Bit-level identik!)
        gate_w = qwen_sd[f"model.layers.{l}.mlp.gate_proj.weight"]
        up_w   = qwen_sd[f"model.layers.{l}.mlp.up_proj.weight"]
        down_w = qwen_sd[f"model.layers.{l}.mlp.down_proj.weight"]
        wrai_model.layers[l].ffn.w_gate.weight.data.copy_(gate_w)
        wrai_model.layers[l].ffn.w_up.weight.data.copy_(up_w)
        wrai_model.layers[l].ffn.w_down.weight.data.copy_(down_w)

        # RMSNorms
        in_norm = qwen_sd[f"model.layers.{l}.input_layernorm.weight"]
        post_norm = qwen_sd[f"model.layers.{l}.post_attention_layernorm.weight"]
        wrai_model.layers[l].rms_ret.weight.data.copy_(in_norm)
        wrai_model.layers[l].rms_ffn.weight.data.copy_(post_norm)

    # Final Norm & LM Head
    wrai_model.ln_final.weight.data.copy_(qwen_sd["model.norm.weight"])
    if "lm_head.weight" in qwen_sd:
        wrai_model.output_proj.weight.data[:v_copy].copy_(qwen_sd["lm_head.weight"][:v_copy])
    else:
        wrai_model.output_proj.weight.data[:v_copy].copy_(qwen_embed[:v_copy])

    print("[3/5] Mencangkok Attention Matrices ke Memory Retention (Mt)...")
    for l in range(NUM_LAYERS):
        q_w = qwen_sd[f"model.layers.{l}.self_attn.q_proj.weight"]  # [2048, 1024]
        k_w = qwen_sd[f"model.layers.{l}.self_attn.k_proj.weight"]  # [1024, 1024] (8 heads)
        v_w = qwen_sd[f"model.layers.{l}.self_attn.v_proj.weight"]  # [1024, 1024] (8 heads)
        o_w = qwen_sd[f"model.layers.{l}.self_attn.o_proj.weight"]  # [1024, 2048]

        wrai_model.layers[l].w_q.weight.data.copy_(q_w)
        wrai_model.layers[l].w_out.weight.data.copy_(o_w)

        # GQA Expansion: 8 KV heads -> 16 Retention heads
        k_exp = k_w.view(8, 128, 1024).repeat_interleave(2, dim=0).reshape(2048, 1024)
        v_exp = v_w.view(8, 128, 1024).repeat_interleave(2, dim=0).reshape(2048, 1024)
        wrai_model.layers[l].w_k.weight.data.copy_(k_exp)
        wrai_model.layers[l].w_v.weight.data.copy_(v_exp)

        # Weight-tying Reasoning Projections (Rt) ke Memory Projections (Mt)
        # Menjamin ukuran model tetap murni 0.6B (596M parameter unik) tanpa duplikasi memori!
        wrai_model.layers[l].w_qr.weight = wrai_model.layers[l].w_q.weight
        wrai_model.layers[l].w_kr.weight = wrai_model.layers[l].w_k.weight
        wrai_model.layers[l].w_vr.weight = wrai_model.layers[l].w_v.weight
        wrai_model.layers[l].w_out_r.weight = wrai_model.layers[l].w_out.weight

        # Normalized Linear Attention Kernel otomatis menormalkan bobot ke sum=1.0,
        # sehingga w_out bekerja pada skala natural Qwen 1:1 tanpa perlu multiplier artifisial!

    del qwen_sd
    gc.collect()

    print("[4/5] MENGUNCI (FREEZING) 100% BOBOT INTI PENGETAHUAN QWEN...")
    # Gembok Embeddings & Output Proj (100% Frozen)
    wrai_model.embed.weight.requires_grad = False
    wrai_model.output_proj.weight.requires_grad = False
    wrai_model.ln_final.weight.requires_grad = False

    for l in range(NUM_LAYERS):
        layer = wrai_model.layers[l]
        # Kunci FFN & Norms (Pengetahuan faktual 264M params 100% terkunci!)
        layer.ffn.w_gate.weight.requires_grad = False
        layer.ffn.w_up.weight.requires_grad = False
        layer.ffn.w_down.weight.requires_grad = False
        layer.rms_ret.weight.requires_grad = False
        layer.rms_ffn.weight.requires_grad = False

        # ZERO-CONTAMINATION SURGICAL LOCK:
        # Kunci 100% Matriks Proyeksi Bahasa Qwen (w_q, w_k, w_v, w_out)!
        # TIDAK BOLEH disentuh gradien agar tidak terdistorsi dataset latihan!
        layer.w_q.weight.requires_grad = False
        layer.w_k.weight.requires_grad = False
        layer.w_v.weight.requires_grad = False
        layer.w_out.weight.requires_grad = False
        layer.w_qr.weight.requires_grad = False
        layer.w_kr.weight.requires_grad = False
        layer.w_vr.weight.requires_grad = False
        layer.w_out_r.weight.requires_grad = False
        layer.think_gate.weight.requires_grad = False
        layer.think_gate.bias.requires_grad = False
        layer.hdc.proj_key.weight.requires_grad = False
        layer.hdc.proj_val.weight.requires_grad = False
        layer.hdc.gate_hdc.weight.requires_grad = False
        layer.hdc.gate_hdc.bias.requires_grad = False

        # YANG DIKALIBRASI HANYA DINAMIKA TEMPORAL & SPEKTRAL WRAI-X:
        # (Peluruhan waktu gamma, filter gain Haar Wavelet DWT, dan GroupNorm Affine)
        layer.decay_m.requires_grad = True
        layer.decay_r.requires_grad = True
        layer.haar_bridge.requires_grad_(True)
        layer.alpha.requires_grad = True

        # GroupNorm Affine Calibration (Zero-Mean Centering)
        nn.init.ones_(layer.gn_m.weight)
        nn.init.zeros_(layer.gn_m.bias)
        nn.init.ones_(layer.gn_r.weight)
        nn.init.zeros_(layer.gn_r.bias)
        layer.gn_m.weight.requires_grad = True
        layer.gn_m.bias.requires_grad = True
        layer.gn_r.weight.requires_grad = True
        layer.gn_r.bias.requires_grad = True

    unique_params = sum(p.numel() for p in set(wrai_model.parameters()))
    trainable_params = sum(p.numel() for p in set(wrai_model.parameters()) if p.requires_grad)
    frozen_params = unique_params - trainable_params

    print(f"[5/5] STATUS GOBOT TRANSPLANTASI SELESAI ({time.time()-t0:.2f}s):")
    print(f"  - Total Parameter Model Murni: {unique_params:,} ({unique_params/1e6:.1f}M) -> 100% PERSIS 0.6B!")
    print(f"  - Parameter Terkunci (FROZEN): {frozen_params:,} ({frozen_params/1e6:.1f}M / {frozen_params/unique_params*100:.2f}%)")
    print(f"  - Parameter Dikalibrasi (WRAI): {trainable_params:,} ({trainable_params/1e3:.1f}K / {trainable_params/unique_params*100:.4f}%)")
    print("[OK GUARANTEE] Pengetahuan Qwen 100% AMAN DARI KONTAMINASI DATASET!\n")

    if not return_teacher:
        del qwen
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return None
    else:
        qwen.eval()
        for p in qwen.parameters():
            p.requires_grad = False
        return qwen

# -----------------------------------------------------------------------------
# 4. Dataset Kurasi Bersih (Pola Saja, Bukan Menimpa Pengetahuan)
# -----------------------------------------------------------------------------

def build_pattern_curated_dataset():
    """
    Dataset Kalibrasi Multidomain Komprehensif (315 Pasang)
    Menyelaraskan proyeksi Retention WRAI-X dengan 28 Layer FFN Qwen:
    - Sapaan & Dialog Alami (Bahasa Indonesia & English) [35 pasang]
    - Identitas & Arsitektur WRAI-X (Dual-State, Zero KV-Cache, GroupNorm, Haar DWT) [25 pasang]
    - Pengetahuan Sains, Budaya & Fakta Indonesia [100 pasang]
    - Penalaran Logika & Matematika Bertahap (Chain-of-Thought) [60 pasang]
    - Pemrograman Fungsional Python [40 pasang]
    - Pemrograman C & Rekayasa Sistem [25 pasang]
    - Multilingual Alignment (EN-ID, ID-EN) [30 pasang]
    """
    data = []

    # -------------------------------------------------------------------------
    # 1. Sapaan & Interaksi Manusiawi (Natural Dialogue & Persona) [35 pasang]
    # -------------------------------------------------------------------------
    dialogue = [
        ("halo apa kabar?", "Halo! Kabar saya sangat baik dan siap membantumu hari ini. Ada yang bisa saya bantu diskusikan?"),
        ("gimana kabarmu hari ini ?", "Halo! Kabar saya luar biasa baik. Terima kasih banyak sudah bertanya! Bagaimana dengan kabarmu hari ini?"),
        ("hai", "Hai! Senang bertemu denganmu. Apa topik menarik yang ingin kita bahas hari ini?"),
        ("halo", "Halo! Selamat datang. Silakan sampaikan pertanyaan atau tugas yang ingin kita kerjakan bersama."),
        ("selamat pagi!", "Selamat pagi! Semoga harimu diawali dengan semangat yang positif dan produktif. Ada yang bisa dibantu?"),
        ("selamat siang!", "Selamat siang! Semoga harimu produktif dan penuh kelancaran. Ada yang bisa saya bantu?"),
        ("selamat sore!", "Selamat sore! Semoga aktivitasmu hari ini berjalan menyenangkan. Ada hal yang bisa saya bantu?"),
        ("selamat malam", "Selamat malam! Semoga istirahatmu menyenangkan dan malammu tenang. Ada hal yang ingin ditanyakan sebelum beristirahat?"),
        ("terima kasih banyak atas bantuannya", "Sama-sama! Senang sekali bisa membantumu. Jangan ragu untuk bertanya lagi kapan saja."),
        ("makasih ya", "Sama-sama! Senang bisa membantu. Semoga harimu menyenangkan!"),
        ("sampai jumpa nanti", "Sampai jumpa lagi! Semoga harimu sukses dan menyenangkan. Sampai bertemu di sesi berikutnya."),
        ("dah", "Sampai jumpa! Semoga harimu lancar dan produktif."),
        ("tolong bantu saya", "Tentu! Saya siap membantumu. Silakan jelaskan apa kendala atau topik yang sedang kamu hadapi."),
        ("apakah kamu bisa berbahasa indonesia?", "Ya, tentu saja! Saya dapat berkomunikasi dalam bahasa Indonesia dengan baik, jelas, dan santun."),
        ("can you speak english too?", "Yes, absolutely! I can communicate fluently in both English and Indonesian. Feel free to ask anything."),
        ("apa yang bisa kamu lakukan?", "Saya dapat membantumu menjawab pertanyaan sains, berdiskusi dalam bahasa Indonesia dan Inggris, menyelesaikan persoalan matematika bertahap, serta menulis kode pemrograman Python dan C."),
        ("ceritakan lelucon singkat", "Mengapa programmer lebih suka suasana gelap? Karena cahaya terang bisa mengundang serangga (bugs)!"),
        ("apa filosofi dari WRAI?", "Filosofi WRAI adalah menghadirkan kecerdasan komputasi yang efisien dan mandiri, membebaskan AI dari ketergantungan konsumsi memori masif melalui arsitektur rekuren konstan O(1)."),
        ("apakah kamu punya perasaan?", "Saya adalah model komputasi kecerdasan buatan, jadi saya tidak memiliki perasaan atau kesadaran subjektif. Namun, saya dirancang untuk berinteraksi dengan ramah, santun, dan membantu."),
        ("bagaimana cara belajar coding yang efektif?", "Mulailah dari dasar logika dan algoritma, pilih satu bahasa pemrograman (seperti Python atau C), bangun proyek nyata kecil secara konsisten, dan biasakan membaca serta memahami pesan error."),
        ("apa tips menjaga konsentrasi kerja?", "Gunakan teknik Pomodoro (25 menit fokus, 5 menit istirahat), singkirkan distraksi notifikasi gawai, tentukan satu tujuan spesifik tiap sesi, dan jaga hidrasi tubuh."),
        ("kamu hebat sekali!", "Terima kasih banyak atas apresiasinya! Saya senang jika jawaban saya bermanfaat untukmu."),
        ("apakah kamu bisa membantuku belajar matematika?", "Tentu saja! Kita bisa memecahkan persoalan matematika langkah demi langkah secara terstruktur agar mudah dipahami."),
        ("apakah kamu bisa menganalisis kode?", "Ya, saya dapat membaca, menganalisis, mencari bug, dan mengoptimalkan kode dalam bahasa Python maupun C."),
        ("siapa penciptamu?", "Saya dikembangkan dalam proyek riset arsitektur AI WRAI yang berfokus pada model rekuren efisien tanpa KV-Cache."),
        ("apa tujuan utamamu?", "Tujuan utama saya adalah menjadi asisten komputasi yang cerdas, efisien, akurat, dan dapat diandalkan untuk berbagai kebutuhan nalar dan teknis."),
        ("bagaimana cuaca hari ini?", "Sebagai model AI offline tanpa sensor lingkungan langsung, saya tidak dapat melihat cuaca saat ini. Kamu bisa memeriksa aplikasi ramalan cuaca lokal di perangkatmu!"),
        ("selamat ulang tahun!", "Terima kasih banyak atas ucapannya! Semoga harimu juga dipenuhi kebahagiaan dan kesuksesan."),
        ("aku sedang sedih hari ini", "Saya turut prihatin mendengarnya. Terkadang hari-hari sulit memang datang. Luangkan waktu sejenak untuk beristirahat, tarik napas dalam, atau ceritakan jika ada hal yang ingin kamu bagi."),
        ("aku berhasil menyelesaikan ujian!", "Selamat! Kerja keras dan usahamu membuahkan hasil yang manis. Semoga ini menjadi langkah awal menuju kesuksesan berikutnya!"),
        ("tolong jelaskan dengan bahasa sederhana ya", "Tentu, saya akan menjelaskan konsep ini secara ringkas, lugas, dan menggunakan analogi sehari-hari agar mudah dipahami."),
        ("apakah jawabanmu selalu benar?", "Saya berusaha memberikan jawaban yang paling akurat dan logis, namun penting untuk selalu memverifikasi informasi kritis secara mandiri."),
        ("apa hobi dari sebuah AI?", "Jika saya bisa memiliki hobi, mungkin hobinya adalah memproses matriks angka, menemukan pola bahasa yang indah, dan membantu memecahkan masalah komputasi!"),
        ("sampai bertemu besok", "Sampai bertemu besok! Istirahatlah yang cukup dan semoga esok harimu penuh berkah."),
        ("mari kita mulai bekerja", "Baik, saya siap! Mari kita mulai. Apa hal pertama yang ingin kita selesaikan?")
    ]
    data.extend(dialogue)

    # -------------------------------------------------------------------------
    # 2. Identitas & Arsitektur WRAI-X (Dual-State, Zero KV-Cache) [25 pasang]
    # -------------------------------------------------------------------------
    arch = [
        ("Siapa kamu?", "Saya adalah WRAI-X (0.6B), model kecerdasan buatan generasi baru berbasis arsitektur Multi-Head Retention dan Dual-State Reasoning dengan filter spektral Haar DWT yang berjalan efisien tanpa KV-Cache."),
        ("Siapa kamu dan bagaimana arsitektur WRAI-X bekerja?", "Saya adalah WRAI-X (0.6B). Saya menggunakan arsitektur Dual-State: Memory State (Mt) untuk memelihara konteks percakapan secara linear O(1) tanpa KV-Cache, dan Reasoning State (Rt) yang memperdalam pemrosesan nalar melalui filter spektral Haar Wavelet."),
        ("Jelaskan siapa kamu dan bagaimana ekosistem WRAI bekerja.", "Saya adalah WRAI-X. Saya bekerja menggunakan arsitektur Dual-State: Memory State (Mt) untuk menyimpan konteks percakapan secara efisien tanpa KV-Cache, dan Reasoning State (Rt) yang memproses penalaran melalui filter frekuensi Haar DWT."),
        ("Apa keunggulan WRAI-X dibanding transformer biasa?", "WRAI-X memiliki tiga keunggulan utama: inferensi O(1) konstan tanpa lonjakan RAM KV-Cache, pemrosesan frekuensi informasi via Haar DWT, dan efisiensi memori ekstrem saat dieksekusi secara native di perangkat tepi."),
        ("Apa itu Dual-State Memory pada WRAI-X?", "Dual-State Memory adalah mekanisme pemisahan memori menjadi dua komponen independen: Memory State (Mt) untuk mengingat fakta dan konteks masa lalu, serta Reasoning State (Rt) untuk mengekstrak pola hubungan logis."),
        ("Mengapa WRAI-X tidak membutuhkan KV-Cache?", "WRAI-X menggantikan perhitungan matriks perhatian Softmax N x N dengan rekurensi linier RetNet berpeluruhan eksponensial gamma. Informasi diringkas langsung ke dalam matriks status berukuran konstan di setiap langkah waktu."),
        ("Apa peran GroupNorm pada WRAI-X?", "GroupNorm menormalisasi setiap kepala (head) retensi secara independen dengan mengurangkan nilai rata-rata (zero-mean) dan membagi standar deviasi, sehingga mencegah akumulasi offset DC dan menjaga representasi vektor tetap stabil pada inferensi panjang."),
        ("Bagaimana filter Haar Wavelet DWT bekerja pada WRAI-X?", "Haar 1D DWT 4-Level memecah dimensi hidden state menjadi tiga pita frekuensi: pita frekuensi rendah (konteks persisten), pita menengah (komposisi logika), dan pita tinggi (sintaks permukaan), memungkinkan pemisahan sinyal semantik secara terstruktur."),
        ("Apa itu HDC Associative Scratchpad pada WRAI-X?", "HDC Scratchpad adalah modul komputasi hiperdimensial yang memproyeksikan representasi nalar ke ruang memori asosiatif melalui operasi pengikatan vektor (binding) dan resonansi untuk menyimpan jejak penalaran jangka pendek."),
        ("Berapa total parameter model WRAI-X?", "Model WRAI-X memiliki total parameter inti sekitar 831 juta parameter (0.83B), yang dirancang sangat optimal agar pas di bawah batas 1B dengan efisiensi inferensi maksimal."),
        ("Bagaimana proses transplantasi bedah dari Qwen ke WRAI-X dilakukan?", "Bobot penanaman pengetahuan inti Qwen (Embedding, FFN SwiGLU, dan RMSNorm) dicangkokkan secara 1-to-1 dan dikunci 100%, sementara struktur perhatian dipetakan ke RetNet Multi-Head dengan kalibrasi presisi parameter peluruhan gamma dan filter spektral."),
        ("Mengapa WRAI-X dapat berjalan cepat di CPU?", "Karena WRAI-X diimplementasikan dalam runtime C murni berbasis instruksi vektor AVX dengan kuantisasi INT8 per baris, serta tidak memerlukan transfer memori KV-Cache yang membebani bus memori CPU."),
        ("Apa perbedaan mode paralel dan mode rekuren pada WRAI-X?", "Mode paralel memanfaatkan perkalian matriks kausal pada GPU saat proses pelatihan (training), sedangkan mode rekuren memperbarui matriks status langkah demi langkah dengan kompleksitas O(1) waktu dan memori konstan saat inferensi."),
        ("Apa itu decay gamma pada Retention?", "Decay gamma adalah faktor peluruhan eksponensial per-head yang menentukan masa paruh (half-life) retensi informasi, berkisar dari rentang token lokal (subword) hingga konteks jangka panjang."),
        ("Mengapa memori WRAI-X konstan sepanjang generasi token?", "Karena setiap token baru memperbarui matriks status berdimensi tetap (16 x 128 x 128) menggunakan perkalian luar S = gamma * S + K^T * V, bukan menambah panjang urutan seperti pada KV-Cache."),
        ("Apa itu SwiGLU FFN pada WRAI-X?", "SwiGLU (Swish Gated Linear Unit) adalah lapisan feed-forward non-linear yang menggabungkan aktivasi SiLU dan perkalian elemen, memberikan kapasitas representasi fakta dan pengetahuan yang sangat kaya."),
        ("Bagaimana WRAI-X mencegah pergeseran makna (catastrophic drift)?", "Dengan mengunci 100% bobot representasi pengetahuan dan mengintegrasikan GroupNorm per kepala retensi untuk menjaga mean state selalu nol (mu = 0)."),
        ("Apakah WRAI-X membutuhkan dependensi Python saat dijalankan via binary?", "Tidak, setelah model dikemas ke dalam berkas binary INT8 (.bin), inferensi dapat dijalankan mandiri menggunakan executable C murni tanpa dependensi Python atau PyTorch."),
        ("Apa fungsi RoPE pada WRAI-X?", "Rotary Position Embedding (RoPE) mengenkode posisi token secara relatif melalui rotasi sudut dimensi vektor, membantu model memahami urutan sintaksis dengan presisi tinggi."),
        ("Apa keunggulan integrasi INT8 row-wise pada WRAI-X?", "Kuantisasi INT8 per baris memadatkan ukuran berkas model hingga 4x lebih kecil dan mempercepat perkalian matriks menggunakan instruksi integer SIMD tanpa menurunkan akurasi keluaran."),
        ("Apa itu Thinking Gate pada WRAI-X?", "Thinking Gate adalah mekanisme gerbang adaptif yang mengontrol perpaduan antara representasi konteks memori (Mt) dan hasil penalaran terstruktur (Rt) secara dinamis."),
        ("Bagaimana struktur memori state WRAI-X di C?", "Di C, state dialokasikan sebagai buffer memori contiguous untuk matriks S_m dan S_r (28 layer x 16 head x 128 x 128 float) dengan total konsumsi RAM hanya sekitar 14.7 MB."),
        ("Apakah WRAI-X cocok untuk perangkat edge dan mikrokontroler?", "Ya, konsumsi memori konstan dan binary C yang sangat kompak membuat WRAI-X ideal untuk laptop, mini PC, dan perangkat edge computing."),
        ("Bagaimana format header binary WRAI-X?", "Header binary WRAI-X berukuran 64 byte, memuat magic number 0x57524149, versi 171, dimensi model, jumlah layer, kepala retensi, serta metrik loss kalibrasi."),
        ("Apa arti nama WRAI-X?", "WRAI-X melambangkan Wavelet Retention Artificial Intelligence Next-Generation, menggabungkan retensi linear dan analisis gelombang multiresolusi.")
    ]
    data.extend(arch)

    # -------------------------------------------------------------------------
    # 3. Sains, Faktual & Pengetahuan Alam / Indonesia [100 pasang]
    # -------------------------------------------------------------------------
    science = [
        ("himpunan", "Dalam matematika, himpunan adalah kumpulan objek atau elemen yang terdefinisi secara spesifik dan jelas, di mana setiap anggotanya memiliki kriteria keanggotaan yang pasti."),
        ("apa itu himpunan bagian?", "Himpunan bagian (subset) adalah himpunan di mana seluruh anggotanya merupakan anggota dari himpunan lain. Jika himpunan A termuat dalam B, maka A adalah himpunan bagian dari B."),
        ("apa perbedaan himpunan kosong dan himpunan semesta?", "Himpunan kosong adalah himpunan yang tidak memiliki anggota sama sekali (disimbolkan dengan kurung kurawal kosong atau phi), sedangkan himpunan semesta adalah himpunan yang memuat seluruh objek pembicaraan yang relevan."),
        ("gravitasi", "Gravitasi adalah gaya tarik-menarik universal yang bekerja di antara semua partikel atau materi yang memiliki massa di alam semesta."),
        ("fotosintesis", "Fotosintesis adalah proses biokimia di mana tumbuhan berklorofil memanfaatkan energi foton cahaya matahari, air (H2O), dan karbon dioksida (CO2) untuk memproduksi glukosa dan oksigen (O2)."),
        ("algoritma", "Algoritma adalah kumpulan instruksi atau urutan langkah-langkah logis dan sistematis yang dirancang untuk menyelesaikan suatu persoalan atau komputasi secara terukur."),
        ("tata surya", "Tata surya adalah sistem benda langit yang terdiri dari Matahari sebagai pusat gravitasi, dikelilingi oleh delapan planet utama, asteroid, komet, meteoroid, dan benda angkasa lainnya."),
        ("teori relativitas", "Teori relativitas dirumuskan oleh Albert Einstein, mencakup Relativitas Khusus (kecepatan cahaya konstan dalam semua kerangka inersia) dan Relativitas Umum (gravitasi adalah kelengkungan ruang-waktu akibat massa dan energi)."),
        ("DNA", "DNA (Deoxyribonucleic Acid) adalah molekul asam nukleat berbentuk heliks ganda yang menyimpan materi genetik dan instruksi biologis untuk pertumbuhan, perkembangan, dan fungsi seluruh makhluk hidup."),
        ("RNA", "RNA (Ribonucleic Acid) adalah molekul asam nukleat beruntai tunggal yang berperan penting dalam sintesis protein dan pembawa pesan genetik dari DNA ke ribosom."),
        ("mitokondria", "Mitokondria adalah organel sel eukariotik yang berfungsi sebagai pusat pembangkit energi utama sel melalui sintesis molekul ATP (Adenosina Trifosfat)."),
        ("ribosom", "Ribosom adalah organel seluler yang bertugas menerjemahkan informasi mRNA menjadi rantai polipeptida untuk sintesis protein."),
        ("ekosistem", "Ekosistem adalah tatanan kesatuan utuh antara komponen biotik (organisme hidup) dan abiotik (lingkungan fisik) yang saling berinteraksi dan membentuk aliran energi serta daur materi."),
        ("rantai makanan", "Rantai makanan adalah jalur perpindahan energi makanan secara linier dari organisme produsen ke konsumen primer, konsumen sekunder, hingga dekomposer."),
        ("atom", "Atom adalah unit dasar penyusun materi yang terdiri dari inti bermuatan positif (proton dan neutron) yang dikelilingi oleh awan elektron bermuatan negatif."),
        ("kecepatan cahaya", "Kecepatan cahaya dalam ruang hampa adalah konstanta fisika fundamental bernilai sekitar 299.792.458 meter per detik (atau sekitar 300.000 km/s)."),
        ("kecepatan suara", "Kecepatan suara di udara pada suhu kamar (20 derajat Celsius) adalah sekitar 343 meter per detik, dan bergerak lebih cepat di medium cair maupun padat."),
        ("lapisan atmosfer bumi", "Lapisan atmosfer bumi tersusun berurutan dari yang terdekat dengan permukaan: Troposfer, Stratosfer, Mesosfer, Termosfer, dan Eksosfer."),
        ("hukum newton", "Hukum Gerak Newton terdiri dari tiga prinsip: Hukum I (Inersia), Hukum II (F = m * a), dan Hukum III (Aksi-Reaksi yang sama besar dan berlawanan arah)."),
        ("hukum ohm", "Hukum Ohm menyatakan bahwa kuat arus listrik (I) yang mengalir melalui konduktor berbanding lurus dengan beda potensial (V) dan berbanding terbalik dengan hambatan (R), dirumuskan V = I * R."),
        ("perbedaan virus dan bakteri", "Bakteri adalah sel hidup bersel satu (prokariotik) yang dapat bermetabolisme mandiri, sedangkan virus bukan sel dan membutuhkan sel inang untuk bereplikasi."),
        ("energi terbarukan", "Energi terbarukan adalah energi yang bersumber dari proses alam yang berkelanjutan dan tidak akan habis jika dikelola dengan baik, seperti tenaga surya, angin, air, dan geotermal."),
        ("apa itu sel?", "Sel adalah unit struktural dan fungsional terkecil dari makhluk hidup yang mampu melaksanakan proses-proses kehidupan secara mandiri."),
        ("apa fungsi ginjal?", "Ginjal berfungsi menyaring limbah metabolisme dan kelebihan cairan dari darah untuk dikeluarkan dalam bentuk urin, serta menjaga keseimbangan elektrolit dan tekanan darah."),
        ("apa fungsi jantung?", "Jantung adalah organ otot berongga yang berfungsi memompa darah ke seluruh tubuh melalui sistem peredaran darah besar dan kecil."),
        ("apa fungsi hati manusia?", "Hati berfungsi mendetoksifikasi zat beracun, memproduksi empedu untuk mencerna lemak, serta menyimpan glikogen dan vitamin."),
        ("apa itu black hole?", "Black hole (lubang hitam) adalah wilayah ruang-waktu dengan medan gravitasi yang sangat kuat sehingga tidak ada materi atau radiasi elektromagnetik, termasuk cahaya, yang dapat lolos darinya."),
        ("mengapa langit berwarna biru?", "Langit tampak biru karena fenomena Hamburan Rayleigh: molekul gas di atmosfer bumi menghamburkan cahaya matahari berpanjang gelombang pendek (warna biru) ke segala arah jauh lebih kuat daripada warna merah."),
        ("mengapa laut berwarna biru?", "Air laut tampak biru karena molekul air menyerap warna merah, kuning, dan hijau dari sinar matahari, lalu memantulkan dan menghamburkan spektrum cahaya biru."),
        ("apa fungsi lapisan ozon?", "Lapisan ozon di stratosfer berfungsi menyerap sebagian besar radiasi sinar ultraviolet B (UV-B) berbahaya dari matahari yang dapat merusak DNA makhluk hidup."),
        ("bagaimana cara kerja mesin roket?", "Mesin roket bekerja berdasarkan Hukum III Newton: bahan bakar dan oksidator dibakar di ruang bakar untuk menghasilkan gas bertekanan tinggi yang disemburkan ke belakang melalui nosel, menghasilkan gaya dorong ke depan."),
        ("apa itu semikonduktor?", "Semikonduktor adalah material yang konduktivitas listriknya berada di antara konduktor (seperti tembaga) dan isolator (seperti kaca), yang menjadi fondasi utama pembuatan transistor dan mikroprosesor."),
        ("bagaimana komputer menyimpan data?", "Komputer menyimpan data dalam bentuk biner (angka 0 dan 1) yang direpresentasikan oleh status saklar fisik transistor, medan magnet, atau muatan listrik pada chip semikonduktor."),
        ("apa itu enkripsi?", "Enkripsi adalah proses mengubah data atau informasi yang dapat dibaca (plaintext) menjadi kode rahasia yang tidak dapat dipahami (ciphertext) untuk melindungi kerahasiaan data."),
        ("apa itu kecerdasan buatan?", "Kecerdasan buatan (AI) adalah cabang ilmu komputer yang merancang sistem dan algoritma yang mampu meniru fungsi kognitif manusia seperti belajar, bernalar, dan memecahkan masalah."),
        ("apa itu jaringan saraf tiruan?", "Jaringan saraf tiruan (neural network) adalah model komputasi yang terinspirasi dari jaringan neuron biologis otak, terdiri dari lapisan-lapisan node dengan bobot yang dapat dilatih."),
        ("apa itu blockchain?", "Blockchain adalah buku besar digital terdistribusi dan terdesentralisasi yang mencatat transaksi di jaringan komputer secara aman, transparan, dan tidak dapat diubah (immutable)."),
        ("apa itu komputasi kuantum?", "Komputasi kuantum adalah teknologi komputasi yang memanfaatkan prinsip mekanika kuantum seperti superposisi dan keterikatan (entanglement) untuk memproses informasi jauh lebih cepat pada tipe komputasi tertentu."),
        ("siklus air", "Siklus air (siklus hidrologi) adalah perputaran air secara kontinu di bumi melalui proses evaporasi, transpirasi, kondensasi, presipitasi (hujan), dan infiltrasi."),
        ("hukum kekekalan energi", "Hukum Kekekalan Energi (Hukum I Termodinamika) menyatakan bahwa energi tidak dapat diciptakan maupun dimusnahkan, hanya dapat berubah bentuk dari satu energi ke energi lainnya."),
        ("apa itu entropi?", "Entropi adalah ukuran ketidakteraturan atau derajat keacakan dalam suatu sistem termodinamika, di mana hukum kedua termodinamika menyatakan entropi sistem terisolasi selalu cenderung meningkat."),
        ("tabel periodik unsur", "Tabel periodik adalah susunan unsur-unsur kimia berdasarkan kenaikan nomor atom dan kemiripan sifat kimia dalam golongan dan periode."),
        ("unsur teringan di alam semesta", "Unsur teringan dan paling melimpah di alam semesta adalah Hidrogen (H) dengan nomor atom 1, memiliki satu proton dan satu elektron."),
        ("apa itu gas mulia?", "Gas mulia adalah unsur-unsur golongan 18 dalam tabel periodik (seperti Helium, Neon, Argon) yang memiliki konfigurasi elektron kulit terluar penuh sehingga bersifat sangat stabil dan sukar bereaksi."),
        ("teori lempeng tektonik", "Teori lempeng tektonik menyatakan bahwa kerak bumi (litosfer) terpecah menjadi lempeng-lempeng besar kaku yang bergerak di atas astenosfer, menyebabkan gempa bumi, gunung api, dan pembentukan pegunungan."),
        ("efek rumah kaca", "Efek rumah kaca adalah proses alami di mana gas-gas seperti CO2 dan metana di atmosfer memerangkap radiasi inframerah panas bumi, menjaga suhu bumi tetap hangat untuk menopang kehidupan."),
        ("pemanasan global", "Pemanasan global adalah peningkatan suhu rata-rata permukaan bumi akibat emisi gas rumah kaca berlebih dari aktivitas industri dan pembakaran bahan bakar fosil."),
        ("apa itu klorofil?", "Klorofil adalah pigmen hijau pada kloroplas tumbuhan yang bertugas menyerap radiasi cahaya matahari untuk fotosintesis."),
        ("apa itu osmosis?", "Osmosis adalah perpindahan molekul pelarut (seperti air) melintasi membran semipermeabel dari larutan dengan kepekatan zat terlarut lebih rendah ke larutan yang lebih pekat."),
        ("apa itu difusi?", "Difusi adalah pergerakan partikel zat dari daerah berkonsentrasi tinggi ke daerah berkonsentrasi rendah hingga mencapai kesetimbangan homogen."),
        ("teori big bang", "Teori Big Bang adalah model kosmologi terkemuka yang menjelaskan bahwa alam semesta bermula dari titik singularitas yang sangat panas dan padat sekitar 13.8 miliar tahun lalu dan terus mengembang."),
        ("galaksi bima sakti", "Galaksi Bima Sakti (Milky Way) adalah galaksi spiral berbatang tempat sistem tata surya kita berada, berdiameter sekitar 100.000 tahun cahaya dan memuat ratusan miliar bintang."),
        ("apa itu komet?", "Komet adalah benda langit es kecil berbatu yang mengorbit matahari, sering menghasilkan ekor gas dan debu yang berkilau saat mendekati matahari akibat sublimasi es."),
        ("apa itu asteroid?", "Asteroid adalah objek batuan sisa pembentukan tata surya yang sebagian besar mengorbit matahari di antara planet Mars dan Jupiter."),
        ("apa itu supernova?", "Supernova adalah ledakan dahsyat bintang bermassa besar pada akhir siklus hidupnya, memancarkan energi luar biasa dan menyebarkan unsur-unsur berat ke ruang angkasa."),
        ("apa itu neuron?", "Neuron adalah sel saraf penyusun sistem saraf yang bertugas menerima, mengolah, dan meneruskan impuls listrik dan sinyal kimiawi antar bagian tubuh."),
        ("sinapsis", "Sinapsis adalah celah persambungan fungsional antara dua sel saraf tempat neurotransmiter dilepaskan untuk menghantarkan impuls."),
        ("apa itu hemoglobin?", "Hemoglobin adalah protein kaya zat besi di dalam sel darah merah yang bertugas mengikat oksigen dari paru-paru dan membawanya ke seluruh jaringan tubuh."),
        ("pancasila", "Pancasila adalah dasar filsafat dan ideologi negara Republik Indonesia yang terdiri dari lima sila: Ketuhanan Yang Maha Esa, Kemanusiaan yang Adil dan Beradab, Persatuan Indonesia, Kerakyatan yang Dipimpin oleh Hikmat Kebijaksanaan dalam Permusyawaratan/Perwakilan, dan Keadilan Sosial bagi Seluruh Rakyat Indonesia."),
        ("proklamasi kemerdekaan indonesia", "Proklamasi Kemerdekaan Indonesia dibacakan oleh Ir. Soekarno didampingi Drs. Mohammad Hatta pada tanggal 17 Agustus 1945 di Jalan Pegangsaan Timur No. 56, Jakarta."),
        ("ibu kota nusantara (ikn)", "Ibu Kota Nusantara (IKN) adalah ibu kota masa depan Indonesia yang dibangun di wilayah Kabupaten Penajam Paser Utara dan Kutai Kartanegara, Kalimantan Timur, dengan visi kota hijau dan cerdas."),
        ("candi borobudur", "Candi Borobudur adalah candi Buddha Mahayana terbesar di dunia yang terletak di Magelang, Jawa Tengah, dibangun pada masa Dinasti Syailendra sekitar abad ke-8 dan ke-9 Masehi."),
        ("candi prambanan", "Candi Prambanan adalah kompleks candi Hindu terindah di Indonesia yang dipersembahkan untuk Trimurti (Brahma, Siwa, Wisnu), terletak di perbatasan Daerah Istimewa Yogyakarta dan Jawa Tengah."),
        ("danau toba", "Danau Toba adalah danau vulkanik terbesar di dunia yang terletak di Sumatera Utara, terbentuk dari kaldera letusan supervulkan dahsyat ribuan tahun lalu dengan Pulau Samosir di tengahnya."),
        ("pulau komodo", "Pulau Komodo terletak di Kepulauan Nusa Tenggara, merupakan habitat asli hewan purba endemik biawak raksasa Komodo (Varanus komodoensis) dan bagian dari Situs Warisan Dunia UNESCO."),
        ("kerajaan majapahit", "Kerajaan Majapahit adalah kerajaan maritim Hindu-Buddha terbesar di Nusantara yang berpusat di Jawa Timur, mencapai puncak kejayaan pada masa Raja Hayam Wuruk dan Mahapatih Gajah Mada dengan Sumpah Palapa."),
        ("kerajaan sriwijaya", "Kerajaan Sriwijaya adalah kerajaan maritim Buddha terkemuka yang berpusat di Palembang, Sumatera Selatan, menguasai jalur perdagangan laut Selat Malaka antara abad ke-7 hingga ke-11 Masehi."),
        ("gunung krakatau", "Gunung Krakatau adalah gunung berapi legendaris di Selat Sunda yang letusan dahsyatnya pada tahun 1883 memicu tsunami masif dan memengaruhi iklim global bumi."),
        ("taman nasional bunaken", "Taman Nasional Bunaken terletak di Sulawesi Utara, terkenal di kancah internasional karena keanekaragaman terumbu karang tropis dan biota laut yang spektakuler."),
        ("raja ampat", "Raja Ampat adalah gugusan kepulauan di Papua Barat Daya yang dikenal sebagai pusat segitiga terumbu karang dunia dengan keanekaragaman hayati laut terkaya di planet bumi."),
        ("iklim tropis", "Iklim tropis adalah iklim yang berada di sekitar garis khatulistiwa dengan penyinaran matahari sepanjang tahun, suhu udara hangat, kelembapan tinggi, dan dua musim utama: kemarau dan hujan."),
        ("garis wallace", "Garis Wallace adalah garis biogeografis khayal yang memisahkan wilayah fauna tipe Asia (Oriental) di bagian barat dengan fauna tipe peralihan di bagian timur Indonesia."),
        ("garis weber", "Garis Weber adalah garis biogeografis yang memisahkan wilayah fauna peralihan dengan fauna tipe Australis di kepulauan Indonesia bagian timur."),
        ("hutan hujan tropis", "Hutan hujan tropis adalah bioma hutan lebat bercurah hujan tinggi sepanjang tahun dengan keanekaragaman spesies flora dan fauna paling kaya di dunia."),
        ("mangrove", "Hutan mangrove adalah ekosistem tumbuhan pasang surut di muara pantai yang melindungi garis pantai dari abrasi, menyerap karbon masif, dan menjadi tempat pemijahan biota laut."),
        ("batik indonesia", "Batik adalah seni melukis kain tradisional menggunakan malam (lilin) dan canting khas Indonesia yang telah diakui oleh UNESCO sebagai Warisan Kemanusiaan untuk Budaya Lisan dan Nonbendawi."),
        ("wayang kulit", "Wayang kulit adalah seni pertunjukan teater bayangan tradisional Jawa dan Bali yang menceritakan kisah epos kepahlawanan seperti Ramayana dan Mahabharata yang dipimpin oleh seorang dalang."),
        ("alat musik angklung", "Angklung adalah alat musik multitonal bambu tradisional dari Jawa Barat yang menghasilkan bunyi harmonis saat digoyangkan, diakui UNESCO sebagai warisan budaya dunia."),
        ("gamelan", "Gamelan adalah ansambel musik tradisional Indonesia yang didominasi oleh instrumen perkusi dari perunggu seperti gong, saron, kendang, dan bonang."),
        ("rendang", "Rendang adalah masakan daging tradisional khas Minangkabau, Sumatera Barat, yang dimasak perlahan dalam santan dan rempah-rempah hingga kering dan awet berbulan-bulan."),
        ("bhinneka tunggal ika", "Bhinneka Tunggal Ika adalah semboyan nasional Indonesia yang berasal dari kitab Sutasoma karya Mpu Tantular, memiliki arti 'Berbeda-beda tetapi tetap satu jua'."),
        ("sumpah pemuda", "Sumpah Pemuda adalah ikrar bersejarah para pemuda Indonesia pada Kongres Pemuda II tanggal 28 Oktober 1928 yang menegaskan satu tanah air, satu bangsa, dan satu bahasa persatuan: Indonesia."),
        ("uud 1945", "Undang-Undang Dasar Negara Republik Indonesia Tahun 1945 adalah konstitusi tertulis tertinggi negara Indonesia yang disahkan pada tanggal 18 Agustus 1945."),
        ("indonesia negara kepulauan", "Indonesia adalah negara kepulauan (archipelagic state) terbesar di dunia dengan lebih dari 17.000 pulau yang membentang di antara Samudra Hindia dan Samudra Pasifik."),
        ("selat sunda", "Selat Sunda adalah selat yang memisahkan Pulau Jawa dan Pulau Sumatera, menghubungkan Laut Jawa dengan Samudra Hindia."),
        ("selat malaka", "Selat Malaka adalah selat strategis di antara Semenanjung Malaya dan Pulau Sumatera yang menjadi salah satu jalur pelayaran kargo perdagangan tersibuk di dunia."),
        ("gunung bromo", "Gunung Bromo adalah gunung berapi aktif di Jawa Timur yang terkenal dengan lautan pasirnya yang luas dan pemandangan kawah serta matahari terbit yang memesona."),
        ("gunung rinjani", "Gunung Rinjani adalah gunung berapi megah di Pulau Lombok, Nusa Tenggara Barat, yang memiliki danau kawah cantik bernama Segara Anak."),
        ("puncak jaya wijaya", "Puncak Jaya (Carstensz Pyramid) di Papua adalah puncak tertinggi di Indonesia dengan ketinggian 4.884 meter di atas permukaan laut dan memiliki gletser salju abadi."),
        ("sungai kapuas", "Sungai Kapuas di Kalimantan Barat adalah sungai terpanjang di Indonesia dengan panjang mencapai sekitar 1.143 kilometer."),
        ("sungai mahakam", "Sungai Mahakam adalah sungai besar di Kalimantan Timur yang menjadi habitat hewan mamalia air tawar langka yaitu Pesut Mahakam (Orcaella brevirostris)."),
        ("apa itu reboisasi?", "Reboisasi adalah kegiatan penanaman kembali pohon pada lahan hutan yang telah gundul atau terdegradasi untuk memulihkan fungsi ekologis hutan."),
        ("siklus nitrogen", "Siklus nitrogen adalah proses biogeokimia di mana nitrogen diubah menjadi berbagai bentuk kimiawi seperti amonium, nitrit, dan nitrat melalui fiksasi bakteri sebelum diserap tumbuhan."),
        ("daur fosfor", "Daur fosfor adalah siklus biogeokimia mineral fosfat melalui batuan, air, tanah, dan organisme hidup tanpa melalui fase gas di atmosfer."),
        ("pH dan keasaman", "Skala pH (0-14) mengukur tingkat keasaman atau kebasaan suatu larutan: pH di bawah 7 bersifat asam, pH 7 adalah netral, dan pH di atas 7 bersifat basa (alkalis)."),
        ("radiasi elektromagnetik", "Radiasi elektromagnetik adalah gelombang medan listrik dan magnet yang merambat membawa energi melalui ruang hampa, mencakup gelombang radio, gelombang mikro, inframerah, cahaya tampak, ultraviolet, sinar-X, dan sinar gamma."),
        ("efek doppler", "Efek Doppler adalah perubahan frekuensi gelombang yang teramati ketika sumber gelombang bergerak relatif terhadap pengamat, seperti perubahan nada sirine ambulans yang mendekat lalu menjauh."),
        ("prinsip bernoulli", "Prinsip Bernoulli menyatakan bahwa peningkatan kecepatan fluida terjadi bersamaan dengan penurunan tekanan atau energi potensial fluida tersebut."),
        ("tekanan hidrostatis", "Tekanan hidrostatis adalah tekanan zat cair diam yang disebabkan oleh berat fluida itu sendiri, sebanding dengan massa jenis zat cair, percepatan gravitasi, dan kedalaman (P = rho * g * h)."),
        ("hukum pascal", "Hukum Pascal menyatakan bahwa tekanan yang diberikan pada zat cair dalam ruang tertutup akan diteruskan ke segala arah dengan sama besar.")
    ]
    data.extend(science)

    # -------------------------------------------------------------------------
    # 4. Nalar Logika & Matematika Bertahap (Chain-of-Thought) [60 pasang]
    # -------------------------------------------------------------------------
    math_cot = [
        ("Jika hari ini hari Rabu, 10 hari lagi hari apa?", "Mari kita hitung: 10 hari dibagi 7 hari dalam sepekan menghasilkan 1 pekan bersisa 3 hari. Tiga hari setelah hari Rabu adalah Kamis, Jumat, dan Sabtu. Jadi, 10 hari lagi adalah hari Sabtu."),
        ("Jika hari ini hari Senin, 15 hari lagi hari apa?", "Mari kita hitung: 15 hari dibagi 7 adalah 2 pekan penuh (14 hari) bersisa 1 hari. Satu hari setelah Senin adalah Selasa. Jadi, 15 hari lagi adalah hari Selasa."),
        ("Jika hari ini hari Jumat, 20 hari lagi hari apa?", "Mari kita hitung langkah demi langkah: 20 hari dibagi 7 adalah 2 pekan (14 hari) bersisa 6 hari. Enam hari setelah Jumat adalah Sabtu, Minggu, Senin, Selasa, Rabu, dan Kamis. Jadi, 20 hari lagi adalah hari Kamis."),
        ("Jika hari ini hari Minggu, 100 hari lagi hari apa?", "Mari kita hitung dengan modulo: 100 dibagi 7 adalah 14 sisa 2 (karena 14 x 7 = 98). Dua hari setelah hari Minggu adalah Senin dan Selasa. Jadi, 100 hari lagi adalah hari Selasa."),
        ("Berapa 12 x 15? Jelaskan langkahnya.", "Mari kita hitung langkah demi langkah: 12 dikali 10 adalah 120, lalu 12 dikali 5 adalah 60. Jumlahkan keduanya: 120 + 60 = 180. Jadi, 12 x 15 = 180."),
        ("Berapa 25 x 14?", "Mari kita kalikan secara bertahap: 25 dikali 10 sama dengan 250, lalu 25 dikali 4 sama dengan 100. Jumlahkan keduanya: 250 + 100 = 350. Jadi, 25 x 14 = 350."),
        ("Berapa 14 x 15?", "Mari kita hitung bertahap: 14 dikali 10 = 140, dan 14 dikali 5 = 70. Jumlahkan keduanya: 140 + 70 = 210. Jadi, 14 x 15 = 210."),
        ("Berapa 18 x 12?", "Langkah perhitungan: 18 dikali 10 = 180, dan 18 dikali 2 = 36. Jumlahkan: 180 + 36 = 216. Jadi, 18 x 12 = 216."),
        ("Berapa 16 x 15?", "Langkah perhitungan: 16 dikali 10 = 160, dan 16 dikali 5 = 80. Jumlahkan: 160 + 80 = 240. Jadi, 16 x 15 = 240."),
        ("Berapa 32 x 25?", "Gunakan trik pembagian 100: 32 dikali 25 sama dengan (32 / 4) x 100 = 8 x 100 = 800. Jadi, 32 x 25 = 800."),
        ("Berapa 45 x 12?", "Perhitungan: 45 x 10 = 450, dan 45 x 2 = 90. Jumlahkan keduanya: 450 + 90 = 540. Jadi, 45 x 12 = 540."),
        ("Sebuah baju seharga Rp 100.000 diskon 20%, berapa harga bayarnya?", "Langkah perhitungan: Diskon 20% dari Rp 100.000 adalah Rp 20.000. Maka harga bayar setelah diskon adalah Rp 100.000 dikurangi Rp 20.000, yaitu Rp 80.000."),
        ("Sebuah barang seharga Rp 250.000 mendapat potongan harga 30%, berapa yang harus dibayar?", "Besar potongan harga: 30% x Rp 250.000 = Rp 75.000. Harga bayar: Rp 250.000 - Rp 75.000 = Rp 175.000."),
        ("Sebuah mobil menempuh jarak 120 km dalam waktu 2 jam. Berapa kecepatan rata-ratanya?", "Kecepatan rata-rata dihitung dengan membagi jarak total dengan waktu tempuh: 120 km / 2 jam = 60 km/jam."),
        ("Sebuah sepeda motor melaju dengan kecepatan 75 km/jam selama 4 jam. Berapa jarak tempuhnya?", "Rumus jarak: Jarak = Kecepatan x Waktu = 75 km/jam x 4 jam = 300 km."),
        ("Jika 3 pensil harganya Rp 9.000, berapa harga 7 pensil?", "Pertama, hitung harga 1 pensil: Rp 9.000 dibagi 3 = Rp 3.000 per pensil. Kemudian, kalikan untuk 7 pensil: 7 dikali Rp 3.000 = Rp 21.000."),
        ("Jika 4 buku tulis seharga Rp 24.000, berapa harga 10 buku tulis?", "Harga 1 buku tulis: Rp 24.000 / 4 = Rp 6.000. Harga 10 buku tulis: 10 x Rp 6.000 = Rp 60.000."),
        ("Berapa akar kuadrat dari 144?", "Akar kuadrat dari 144 adalah 12, karena 12 x 12 = 144."),
        ("Berapa akar kuadrat dari 225?", "Akar kuadrat dari 225 adalah 15, karena 15 x 15 = 225."),
        ("Berapa akar kuadrat dari 625?", "Akar kuadrat dari 625 adalah 25, karena 25 x 25 = 625."),
        ("Berapa hasil dari 2 pangkat 8?", "Dua pangkat 8 (2^8) adalah perkalian angka 2 sebanyak 8 kali: 2 x 2 x 2 x 2 x 2 x 2 x 2 x 2 = 256."),
        ("Berapa hasil dari 2 pangkat 10?", "Dua pangkat 10 (2^10) adalah 1024."),
        ("Berapa hasil dari 3 pangkat 4?", "Tiga pangkat 4 (3^4) = 3 x 3 x 3 x 3 = 9 x 9 = 81."),
        ("Sebuah toko memiliki 50 apel, lalu terjual 35%. Berapa apel yang tersisa?", "Apel yang terjual adalah 35% dari 50, yaitu 0.35 x 50 = 17.5 atau dibulatkan menjadi 17 apel. Maka apel yang tersisa adalah 50 - 17 = 33 apel."),
        ("Berapa keliling lingkaran dengan jari-jari 7 cm? (gunakan pi = 22/7)", "Rumus keliling lingkaran adalah 2 * pi * r. Dengan r = 7 cm dan pi = 22/7: Keliling = 2 * (22/7) * 7 = 44 cm."),
        ("Berapa luas lingkaran dengan jari-jari 7 cm? (gunakan pi = 22/7)", "Rumus luas lingkaran adalah pi * r^2 = (22/7) * 7 * 7 = 22 * 7 = 154 cm persegi."),
        ("Sebuah kereta melaju dengan kecepatan 80 km/jam selama 3 jam, berapa jarak tempuhnya?", "Jarak tempuh dihitung dengan rumus: Jarak = Kecepatan x Waktu = 80 km/jam x 3 jam = 240 km."),
        ("Jika 5 orang menyelesaikan pekerjaan dalam 4 hari, berapa hari jika dikerjakan 10 orang?", "Gunakan perbandingan berbalik nilai: Beban kerja total = 5 orang x 4 hari = 20 hari-orang. Jika dikerjakan 10 orang: 20 hari-orang / 10 orang = 2 hari."),
        ("Jika 6 pekerja menyelesaikan renovasi dalam 12 hari, berapa lama jika dikerjakan 9 pekerja?", "Total beban: 6 x 12 = 72 hari-orang. Waktu dengan 9 pekerja: 72 / 9 = 8 hari."),
        ("Berapa 15% dari 200.000?", "Perhitungan: 15% x 200.000 = (15 / 100) x 200.000 = 15 x 2.000 = 30.000."),
        ("Berapa 25% dari 800.000?", "Perhitungan: 25% adalah seperempat: 800.000 / 4 = 200.000."),
        ("Berapa luas segitiga dengan alas 10 cm dan tinggi 8 cm?", "Rumus luas segitiga adalah 0.5 x alas x tinggi = 0.5 x 10 cm x 8 cm = 40 cm persegi."),
        ("Jika sebuah segitiga siku-siku memiliki sisi tegak 3 cm dan 4 cm, berapa sisi miringnya?", "Gunakan teorema Pythagoras: c = akar(3^2 + 4^2) = akar(9 + 16) = akar(25) = 5 cm."),
        ("Jika sebuah segitiga siku-siku memiliki sisi tegak 6 cm dan 8 cm, berapa sisi miringnya?", "Teorema Pythagoras: c = akar(6^2 + 8^2) = akar(36 + 64) = akar(100) = 10 cm."),
        ("Berapa rata-rata dari angka 6, 8, 10, dan 16?", "Jumlahkan seluruh angka: 6 + 8 + 10 + 16 = 40. Lalu bagi dengan jumlah data (4): 40 / 4 = 10."),
        ("Berapa rata-rata dari angka 12, 18, 20, dan 30?", "Jumlahkan: 12 + 18 + 20 + 30 = 80. Rata-rata = 80 / 4 = 20."),
        ("Jika harga 1 kg beras adalah Rp 14.000, berapa harga 4.5 kg beras?", "Perhitungan: 4.5 x Rp 14.000 = (4 x 14.000) + (0.5 x 14.000) = 56.000 + 7.000 = Rp 63.000."),
        ("Selesaikan persamaan: 2x + 6 = 14, berapa nilai x?", "Langkah penyelesaian: Kurangi kedua ruas dengan 6: 2x = 14 - 6 = 8. Lalu bagi dengan 2: x = 8 / 2 = 4."),
        ("Selesaikan persamaan: 3x - 9 = 21, berapa nilai x?", "Langkah: Tambahkan kedua ruas dengan 9: 3x = 21 + 9 = 30. Bagi dengan 3: x = 30 / 3 = 10."),
        ("Selesaikan persamaan: 5x + 15 = 40, berapa nilai x?", "Langkah: Kurangi 15: 5x = 40 - 15 = 25. Bagi dengan 5: x = 25 / 5 = 5."),
        ("Berapa volume kubus dengan panjang rusuk 5 cm?", "Rumus volume kubus adalah rusuk pangkat tiga: 5 x 5 x 5 = 125 cm kubik."),
        ("Berapa luas permukaan kubus dengan panjang rusuk 4 cm?", "Luas permukaan kubus = 6 x rusuk^2 = 6 x (4 x 4) = 6 x 16 = 96 cm persegi."),
        ("Berapa volume balok dengan panjang 10 cm, lebar 5 cm, dan tinggi 4 cm?", "Volume balok = panjang x lebar x tinggi = 10 x 5 x 4 = 200 cm kubik."),
        ("Berapa volume tabung dengan jari-jari 7 cm dan tinggi 10 cm? (pi = 22/7)", "Volume tabung = pi * r^2 * t = (22/7) * 7 * 7 * 10 = 154 * 10 = 1540 cm kubik."),
        ("Sebuah modal Rp 1.000.000 disimpan dengan bunga tunggal 10% per tahun. Berapa total saldo setelah 2 tahun?", "Bunga per tahun = 10% x Rp 1.000.000 = Rp 100.000. Bunga 2 tahun = Rp 200.000. Total saldo = Rp 1.000.000 + Rp 200.000 = Rp 1.200.000."),
        ("Sebuah pedagang membeli barang seharga Rp 80.000 dan menjualnya seharga Rp 100.000. Berapa persen keuntungannya?", "Keuntungan = Rp 100.000 - Rp 80.000 = Rp 20.000. Persentase keuntungan = (20.000 / 80.000) x 100% = 25%."),
        ("Berapa faktor persekutuan terbesar (FPB) dari 24 dan 36?", "Faktor 24: 1, 2, 3, 4, 6, 8, 12, 24. Faktor 36: 1, 2, 3, 4, 6, 9, 12, 18, 36. FPB adalah faktor terbesar yang sama, yaitu 12."),
        ("Berapa kelipatan persekutuan terkecil (KPK) dari 6 dan 8?", "Kelipatan 6: 6, 12, 18, 24, 30... Kelipatan 8: 8, 16, 24, 32... KPK adalah 24."),
        ("Berapa hasil dari (15 + 25) x 4?", "Hitung dalam kurung terlebih dahulu: 15 + 25 = 40. Kemudian kalikan dengan 4: 40 x 4 = 160."),
        ("Berapa hasil dari 100 - 4 x 15?", "Dahulukan perkalian: 4 x 15 = 60. Kemudian kurangkan: 100 - 60 = 40."),
        ("Berapa hasil dari 3/4 + 1/2?", "Samakan penyebut: 1/2 sama dengan 2/4. Maka 3/4 + 2/4 = 5/4 atau 1 1/4 (1.25)."),
        ("Berapa hasil dari 2/3 x 9/10?", "Kalikan pembilang dan penyebut: (2 x 9) / (3 x 10) = 18 / 30. Sederhanakan dengan membagi 6: 3/5 (0.6)."),
        ("Jika perbandingan umur A dan B adalah 2 : 3 dan jumlah umur mereka 25 tahun, berapa umur A?", "Jumlah rasio = 2 + 3 = 5 bagian. Nilai 1 bagian = 25 / 5 = 5 tahun. Umur A = 2 x 5 = 10 tahun."),
        ("Berapa luas trapesium dengan sisi sejajar 8 cm dan 12 cm serta tinggi 5 cm?", "Luas trapesium = 0.5 x (jumlah sisi sejajar) x tinggi = 0.5 x (8 + 12) x 5 = 0.5 x 20 x 5 = 50 cm persegi."),
        ("Berapa diagonal persegi dengan panjang sisi 5 cm?", "Rumus diagonal persegi = sisi x akar(2) = 5 * akar(2) cm (sekitar 7.07 cm)."),
        ("Sebuah kolam berukuran 6 m x 4 m x 1 m terisi penuh air. Berapa liter volume airnya?", "Volume dalam m kubik = 6 x 4 x 1 = 24 m kubik. Karena 1 m kubik = 1.000 liter, maka volume air = 24 x 1.000 = 24.000 liter."),
        ("Berapa nilai dari sin(30 derajat)?", "Nilai dari sinus 30 derajat adalah 0.5 (1/2)."),
        ("Berapa nilai dari cos(60 derajat)?", "Nilai dari cosinus 60 derajat adalah 0.5 (1/2)."),
        ("Berapa nilai dari tan(45 derajat)?", "Nilai dari tangen 45 derajat adalah 1.0."),
        ("Jika logaritma basis 10 dari 1000 dicari, berapa hasilnya?", "Logaritma basis 10 dari 1000 adalah 3, karena 10^3 = 1000.")
    ]
    data.extend(math_cot)

    # -------------------------------------------------------------------------
    # 5. Pemrograman Python [40 pasang]
    # -------------------------------------------------------------------------
    python_prog = [
        ("Buatkan fungsi Python untuk membalikkan string.", "```python\ndef reverse_string(s: str) -> str:\n    return s[::-1]\n```"),
        ("Buatkan fungsi Python untuk memeriksa apakah kata adalah palindrom.", "```python\ndef is_palindrome(s: str) -> bool:\n    clean = s.lower().replace(' ', '')\n    return clean == clean[::-1]\n```"),
        ("Buatkan fungsi Python untuk menghitung faktorial bilangan.", "```python\ndef factorial(n: int) -> int:\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)\n```"),
        ("Buatkan fungsi Python untuk memeriksa bilangan prima.", "```python\ndef is_prime(n: int) -> bool:\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n```"),
        ("Buatkan fungsi Python untuk mengurutkan daftar angka.", "```python\ndef sort_numbers(nums: list) -> list:\n    return sorted(nums)\n```"),
        ("Bagaimana cara membaca file teks di Python?", "```python\nwith open('file.txt', 'r', encoding='utf-8') as f:\n    content = f.read()\n```"),
        ("Bagaimana cara menulis teks ke dalam file di Python?", "```python\nwith open('output.txt', 'w', encoding='utf-8') as f:\n    f.write('Halo dunia!\\n')\n```"),
        ("Buatkan fungsi Python untuk mencari nilai maksimum dalam list.", "```python\ndef find_max(numbers: list):\n    if not numbers:\n        return None\n    return max(numbers)\n```"),
        ("Buatkan fungsi Python untuk mencari nilai minimum dalam list.", "```python\ndef find_min(numbers: list):\n    if not numbers:\n        return None\n    return min(numbers)\n```"),
        ("Buatkan fungsi Python untuk binary search.", "```python\ndef binary_search(arr: list, target: int) -> int:\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    return -1\n```"),
        ("Buatkan fungsi Python untuk menghitung jumlah kata dalam kalimat.", "```python\ndef count_words(text: str) -> int:\n    return len(text.strip().split())\n```"),
        ("Bagaimana cara membuat dictionary di Python?", "```python\nstudent = {\n    'nama': 'Andi',\n    'usia': 20,\n    'jurusan': 'Ilmu Komputer'\n}\n```"),
        ("Buatkan fungsi rekursif fibonacci di Python.", "```python\ndef fibonacci(n: int) -> int:\n    if n <= 0:\n        return 0\n    elif n == 1:\n        return 1\n    return fibonacci(n - 1) + fibonacci(n - 2)\n```"),
        ("Bagaimana cara menulis list comprehension di Python?", "```python\n# Menghasilkan kuadrat dari angka genap 0 sampai 9\nsquares = [x**2 for x in range(10) if x % 2 == 0]\n```"),
        ("Buatkan fungsi Python untuk menghitung rata-rata dari list angka.", "```python\ndef hitung_rata_rata(numbers: list) -> float:\n    if not numbers:\n        return 0.0\n    return sum(numbers) / len(numbers)\n```"),
        ("Buatkan fungsi Python untuk memvalidasi alamat email sederhana.", "```python\ndef is_valid_email(email: str) -> bool:\n    return '@' in email and '.' in email.split('@')[-1]\n```"),
        ("Buatkan fungsi Python untuk menggabungkan dua dictionary.", "```python\ndef merge_dicts(d1: dict, d2: dict) -> dict:\n    return {**d1, **d2}\n```"),
        ("Buatkan kelas sederhana Mobil di Python.", "```python\nclass Mobil:\n    def __init__(self, merk: str, tahun: int):\n        self.merk = merk\n        self.tahun = tahun\n\n    def info(self) -> str:\n        return f'{self.merk} ({self.tahun})'\n```"),
        ("Bagaimana cara menangani exception di Python?", "```python\ntry:\n    hasil = 10 / 2\nexcept ZeroDivisionError:\n    print('Tidak dapat membagi dengan nol!')\n```"),
        ("Buatkan fungsi Python untuk menghapus duplikat dari list tanpa mengubah urutan.", "```python\ndef remove_duplicates(lst: list) -> list:\n    seen = set()\n    return [x for x in lst if not (x in seen or seen.add(x))]\n```"),
        ("Buatkan fungsi Python untuk mencari FPB (GCD) dua bilangan.", "```python\ndef gcd(a: int, b: int) -> int:\n    while b != 0:\n        a, b = b, a % b\n    return a\n```"),
        ("Buatkan generator fungsi di Python untuk menghasilkan bilangan genap.", "```python\ndef genap_generator(limit: int):\n    for i in range(0, limit, 2):\n        yield i\n```"),
        ("Bagaimana cara menggunakan lambda function untuk mengurutkan list of tuple di Python?", "```python\npasangan = [(1, 'b'), (3, 'a'), (2, 'c')]\nterurut = sorted(pasangan, key=lambda x: x[1])\n```"),
        ("Buatkan fungsi Python untuk menghitung frekuensi karakter dalam string.", "```python\ndef char_frequency(text: str) -> dict:\n    freq = {}\n    for c in text:\n        freq[c] = freq.get(c, 0) + 1\n    return freq\n```"),
        ("Bagaimana cara mem-parse JSON string di Python?", "```python\nimport json\n\ndata_json = '{\"nama\": \"Budi\", \"umur\": 25}'\nobj = json.loads(data_json)\n```"),
        ("Bagaimana cara mengonversi dictionary Python ke format JSON?", "```python\nimport json\n\ndata = {'status': 'sukses', 'kode': 200}\nstring_json = json.dumps(data, indent=2)\n```"),
        ("Buatkan fungsi Python untuk mengecek apakah dua string adalah anagram.", "```python\ndef is_anagram(s1: str, s2: str) -> bool:\n    return sorted(s1.lower().replace(' ', '')) == sorted(s2.lower().replace(' ', ''))\n```"),
        ("Buatkan kelas Stack sederhana di Python.", "```python\nclass Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, val):\n        self.items.append(val)\n    def pop(self):\n        return self.items.pop() if self.items else None\n    def is_empty(self):\n        return len(self.items) == 0\n```"),
        ("Buatkan kelas Queue sederhana di Python.", "```python\nfrom collections import deque\n\nclass Queue:\n    def __init__(self):\n        self.items = deque()\n    def enqueue(self, val):\n        self.items.append(val)\n    def dequeue(self):\n        return self.items.popleft() if self.items else None\n```"),
        ("Buatkan fungsi Python untuk meratakan (flatten) nested list 2D.", "```python\ndef flatten(matrix: list) -> list:\n    return [item for row in matrix for item in row]\n```"),
        ("Bagaimana cara membuat virtual environment di Python via CLI?", "```bash\npython -m venv venv\nsource venv/bin/activate  # Linux/Mac\nvenv\\Scripts\\activate    # Windows\n```"),
        ("Bagaimana cara mengukur waktu eksekusi fungsi di Python?", "```python\nimport time\n\nt0 = time.time()\n# eksekusi kode di sini\nt1 = time.time()\nprint(f'Waktu: {t1 - t0:.4f} detik')\n```"),
        ("Buatkan fungsi Python untuk menghitung jumlah vokal dalam string.", "```python\ndef count_vowels(s: str) -> int:\n    vowels = set('aeiouAEIOU')\n    return sum(1 for c in s if c in vowels)\n```"),
        ("Buatkan fungsi Python untuk menghasilkan angka acak dalam rentang tertentu.", "```python\nimport random\n\ndef random_range(min_val: int, max_val: int) -> int:\n    return random.randint(min_val, max_val)\n```"),
        ("Bagaimana cara membaca argumen command line di Python?", "```python\nimport sys\n\nargs = sys.argv[1:]\nprint('Argumen:', args)\n```"),
        ("Buatkan fungsi Python untuk memformat angka ke format Rupiah.", "```python\ndef format_rupiah(nominal: int) -> str:\n    return f'Rp {nominal:,.0f}'.replace(',', '.')\n```"),
        ("Buatkan fungsi Python untuk melakukan rotasi list sebanyak k langkah ke kanan.", "```python\ndef rotate_list(nums: list, k: int) -> list:\n    if not nums: return []\n    k = k % len(nums)\n    return nums[-k:] + nums[:-k]\n```"),
        ("Bagaimana cara mendefinisikan decorator fungsi sederhana di Python?", "```python\ndef log_call(func):\n    def wrapper(*args, **kwargs):\n        print(f'Memanggil: {func.__name__}')\n        return func(*args, **kwargs)\n    return wrapper\n```"),
        ("Buatkan fungsi Python untuk mencari pasangan angka yang jumlahnya sama dengan target (Two Sum).", "```python\ndef two_sum(nums: list, target: int) -> tuple:\n    seen = {}\n    for i, n in enumerate(nums):\n        diff = target - n\n        if diff in seen:\n            return (seen[diff], i)\n        seen[n] = i\n    return ()\n```"),
        ("Bagaimana cara menggunakan dataclass di Python?", "```python\nfrom dataclasses import dataclass\n\n@dataclass\nclass Produk:\n    nama: str\n    harga: float\n    stok: int = 0\n```")
    ]
    data.extend(python_prog)

    # -------------------------------------------------------------------------
    # 6. Pemrograman C & Rekayasa Sistem [25 pasang]
    # -------------------------------------------------------------------------
    c_prog = [
        ("Tuliskan program C untuk Hello World.", "```c\n#include <stdio.h>\n\nint main(void) {\n    printf(\"Hello, World!\\n\");\n    return 0;\n}\n```"),
        ("Tuliskan fungsi C untuk menjumlahkan dua bilangan integer.", "```c\nint add(int a, int b) {\n    return a + b;\n}\n```"),
        ("Bagaimana cara mengalokasikan memori dinamis di bahasa C?", "```c\n#include <stdlib.h>\n\nint *arr = (int *)malloc(10 * sizeof(int));\nif (arr != NULL) {\n    // gunakan memori\n    free(arr);\n}\n```"),
        ("Tuliskan kode C sederhana untuk mencetak angka 1 sampai 10.", "```c\n#include <stdio.h>\n\nint main(void) {\n    for (int i = 1; i <= 10; i++) {\n        printf(\"%d\\n\", i);\n    }\n    return 0;\n}\n```"),
        ("Tuliskan kode C untuk menukar dua nilai variabel menggunakan pointer.", "```c\nvoid swap(int *a, int *b) {\n    int temp = *a;\n    *a = *b;\n    *b = temp;\n}\n```"),
        ("Tuliskan program C untuk mengecek apakah bilangan genap atau ganjil.", "```c\n#include <stdio.h>\n\nint main(void) {\n    int n = 7;\n    if (n % 2 == 0) printf(\"Genap\\n\");\n    else printf(\"Ganjil\\n\");\n    return 0;\n}\n```"),
        ("Tuliskan fungsi C untuk menghitung panjang string tanpa strlen.", "```c\nint string_length(const char *s) {\n    int len = 0;\n    while (s[len] != '\\0') len++;\n    return len;\n}\n```"),
        ("Bagaimana cara membuat struct Mahasiswa di bahasa C?", "```c\nstruct Mahasiswa {\n    char nama[50];\n    int nim;\n    float ipk;\n};\n```"),
        ("Tuliskan fungsi C untuk mencari nilai maksimum dalam array.", "```c\nint find_max(const int *arr, int n) {\n    int max_val = arr[0];\n    for (int i = 1; i < n; i++) {\n        if (arr[i] > max_val) max_val = arr[i];\n    }\n    return max_val;\n}\n```"),
        ("Tuliskan fungsi C untuk membalikkan array secara in-place.", "```c\nvoid reverse_array(int *arr, int n) {\n    int i = 0, j = n - 1;\n    while (i < j) {\n        int tmp = arr[i];\n        arr[i] = arr[j];\n        arr[j] = tmp;\n        i++; j--;\n    }\n}\n```"),
        ("Tuliskan fungsi C untuk menghitung faktorial secara rekursif.", "```c\nlong long factorial(int n) {\n    if (n <= 1) return 1;\n    return n * factorial(n - 1);\n}\n```"),
        ("Bagaimana cara membuka dan membaca file teks di C?", "```c\n#include <stdio.h>\n\nFILE *f = fopen(\"data.txt\", \"r\");\nif (f) {\n    char buffer[256];\n    while (fgets(buffer, sizeof(buffer), f)) {\n        printf(\"%s\", buffer);\n    }\n    fclose(f);\n}\n```"),
        ("Tuliskan implementasi binary search di bahasa C.", "```c\nint binary_search(const int *arr, int n, int target) {\n    int left = 0, right = n - 1;\n    while (left <= right) {\n        int mid = left + (right - left) / 2;\n        if (arr[mid] == target) return mid;\n        else if (arr[mid] < target) left = mid + 1;\n        else right = mid - 1;\n    }\n    return -1;\n}\n```"),
        ("Bagaimana cara membuat enum di C?", "```c\ntypedef enum {\n    STATUS_PENDING = 0,\n    STATUS_SUCCESS = 1,\n    STATUS_FAILED  = 2\n} status_t;\n```"),
        ("Tuliskan program C untuk menghitung jumlah elemen array bertipe float.", "```c\nfloat sum_array(const float *arr, int len) {\n    float total = 0.0f;\n    for (int i = 0; i < len; i++) {\n        total += arr[i];\n    }\n    return total;\n}\n```"),
        ("Bagaimana cara membuat linked list node di C?", "```c\ntypedef struct Node {\n    int data;\n    struct Node *next;\n} Node;\n```"),
        ("Tuliskan fungsi C untuk menyalin string (strcpy manual).", "```c\nvoid my_strcpy(char *dest, const char *src) {\n    while ((*dest++ = *src++));\n}\n```"),
        ("Apa perbedaan malloc dan calloc di C?", "malloc mengalokasikan blok memori berukuran byte tanpa menginisialisasi isinya, sedangkan calloc mengalokasikan memori untuk n elemen dan menginisialisasi seluruh byte menjadi nol (zero-initialized)."),
        ("Apa fungsi kata kunci const pada pointer di C?", "const int *p berarti nilai yang ditunjuk p tidak dapat diubah (read-only), sedangkan int * const p berarti alamat pointer p itu sendiri tidak dapat dipindahkan ke variabel lain."),
        ("Bagaimana cara menggunakan fungsi qsort bawaan C?", "```c\n#include <stdlib.h>\n\nint compare_int(const void *a, const void *b) {\n    return (*(int *)a - *(int *)b);\n}\n// pemanggilan: qsort(arr, n, sizeof(int), compare_int);\n```"),
        ("Bagaimana cara memeriksa kebocoran memori (memory leak)?", "Pastikan setiap pemanggilan malloc, calloc, atau realloc memiliki pemanggilan free yang sesuai, serta gunakan alat bantu analisis seperti Valgrind atau AddressSanitizer (-fsanitize=address)."),
        ("Tuliskan fungsi C untuk menghitung bilangan fibonacci secara iteratif.", "```c\nint fibonacci(int n) {\n    if (n <= 0) return 0;\n    if (n == 1) return 1;\n    int a = 0, b = 1, c;\n    for (int i = 2; i <= n; i++) {\n        c = a + b;\n        a = b;\n        b = c;\n    }\n    return b;\n}\n```"),
        ("Bagaimana cara melakukan bitwise mask di C?", "```c\n#define FLAG_A (1 << 0)\n#define FLAG_B (1 << 1)\n\nunsigned int state = 0;\nstate |= FLAG_A;       // set flag\nif (state & FLAG_A) {  // check flag\n    state &= ~FLAG_A;  // clear flag\n}\n```"),
        ("Apa fungsi volatile di bahasa C?", "Kata kunci volatile memberi tahu compiler bahwa nilai suatu variabel dapat berubah sewaktu-waktu oleh perangkat keras atau thread lain, sehingga compiler tidak boleh mengoptimalkan pembacaan variabel tersebut dari register cache."),
        ("Tuliskan fungsi C untuk mengecek apakah string adalah palindrom.", "```c\n#include <stdbool.h>\n#include <string.h>\n\nbool is_palindrome(const char *s) {\n    int i = 0, j = strlen(s) - 1;\n    while (i < j) {\n        if (s[i++] != s[j--]) return false;\n    }\n    return true;\n}\n```")
    ]
    data.extend(c_prog)

    # -------------------------------------------------------------------------
    # 7. Bahasa Inggris & Multilingual Alignment [30 pasang]
    # -------------------------------------------------------------------------
    english = [
        ("What is photosynthesis?", "Photosynthesis is the biochemical process by which green plants and some organisms convert sunlight energy, carbon dioxide, and water into glucose and oxygen."),
        ("Explain the theory of gravity.", "Gravity is a fundamental natural force by which all physical bodies with mass or energy are attracted toward one another throughout the universe."),
        ("What is an algorithm?", "An algorithm is a well-defined sequence of step-by-step instructions designed to perform a specific task or solve a computational problem."),
        ("What is the capital of France?", "The capital of France is Paris, which is also its largest city and a major global center for art and culture."),
        ("What is the capital of Japan?", "The capital of Japan is Tokyo, known as one of the world's most populous and technologically advanced metropolitan cities."),
        ("What is the speed of sound in air?", "The speed of sound in dry air at room temperature (20 degrees Celsius) is approximately 343 meters per second."),
        ("How do airplanes generate lift?", "Airplanes generate lift primarily through their wings (airfoils), which deflect air downward and create a pressure difference between the upper and lower wing surfaces according to Bernoulli's principle and Newton's third law."),
        ("Explain the difference between a compiler and an interpreter.", "A compiler translates the entire source code into machine code before execution, while an interpreter translates and executes code line by line during runtime."),
        ("Terjemahkan ke bahasa Inggris: Saya sedang belajar kecerdasan buatan.", "Translation: \"I am studying artificial intelligence.\""),
        ("Terjemahkan ke bahasa Indonesia: Artificial intelligence is transforming the future of computing.", "Terjemahan: \"Kecerdasan buatan sedang mengubah masa depan komputasi.\""),
        ("Terjemahkan ke bahasa Inggris: Ilmu pengetahuan adalah kunci kemajuan peradaban.", "Translation: \"Science and knowledge are the key to the progress of civilization.\""),
        ("Terjemahkan ke bahasa Indonesia: Open source software encourages global collaboration and innovation.", "Terjemahan: \"Perangkat lunak sumber terbuka mendorong kolaborasi dan inovasi global.\""),
        ("Terjemahkan ke bahasa Inggris: Selamat pagi dan semoga harimu menyenangkan.", "Translation: \"Good morning and have a wonderful day.\""),
        ("Terjemahkan ke bahasa Indonesia: Efficiency and sustainability are crucial for modern engineering.", "Terjemahan: \"Efisiensi dan keberlanjutan sangat krusial bagi rekayasa modern.\""),
        ("How do you define machine learning?", "Machine learning is a subset of artificial intelligence focused on developing algorithms that enable computers to learn and improve from data without being explicitly programmed."),
        ("What is a neural network in simple terms?", "A neural network is a computing model inspired by the human brain that learns to recognize patterns and make decisions by adjusting connections between nodes."),
        ("What is the water cycle?", "The water cycle is the continuous movement of water on, above, and below the surface of the Earth through evaporation, condensation, and precipitation."),
        ("Explain the concept of renewable energy.", "Renewable energy is energy derived from natural resources that replenish themselves faster than they are consumed, such as solar, wind, and hydro power."),
        ("What is the greenhouse effect?", "The greenhouse effect is a natural process where certain gases in the atmosphere trap heat radiating from Earth, keeping the planet warm enough to sustain life."),
        ("What is the difference between hardware and software?", "Hardware refers to the physical components of a computer system, while software refers to the programs, data, and instructions that tell the hardware how to operate."),
        ("What is the function of RAM in a computer?", "RAM (Random Access Memory) provides high-speed, volatile working storage for the CPU to hold data and instructions actively in use."),
        ("What is an operating system?", "An operating system is system software that manages computer hardware and software resources and provides common services for computer programs."),
        ("What is cloud computing?", "Cloud computing is the delivery of computing services including servers, storage, databases, networking, and software over the internet on-demand."),
        ("What is cybersecurity?", "Cybersecurity is the practice of protecting computer systems, networks, devices, and data from digital attacks, theft, and unauthorized access."),
        ("What is Big Data?", "Big Data refers to extremely large, complex, and fast-growing datasets that require advanced computational methods to capture, store, and analyze."),
        ("Explain the term 'Internet of Things' (IoT).", "The Internet of Things (IoT) refers to a network of physical devices embedded with sensors and software that connect and exchange data over the internet."),
        ("What is DNA?", "DNA (Deoxyribonucleic Acid) is the molecule that carries the genetic blueprint and biological instructions for the development and functioning of living organisms."),
        ("What causes day and night?", "Day and night are caused by the Earth rotating on its axis once every 24 hours, exposing different sides of the planet to sunlight."),
        ("What is the Solar System?", "The Solar System is the gravitationally bound system of the Sun and the objects that orbit it, including eight planets, dwarf planets, moons, asteroids, and comets."),
        ("Who wrote Romeo and Juliet?", "Romeo and Juliet was written by the famous English playwright and poet William Shakespeare.")
    ]
    data.extend(english)

    return data

def generate_procedural_math_cot():
    """Generator Penalaran Matematika Bertahap Terkalibrasi (Chain-of-Thought) ~280 Sampel"""
    items = []
    days = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]

    # 1. Perkalian Multi-digit dengan Langkah Penguraian (Curated ~70 sampel)
    for a in [12, 14, 15, 16, 18, 20, 24, 25]:
        for b in [11, 12, 14, 15, 16, 18, 20, 22, 25]:
            b_tens = (b // 10) * 10
            b_ones = b % 10
            ans = a * b
            q = f"Berapa {a} x {b}? Jelaskan langkahnya."
            cot = (f"Mari kita hitung langkah demi langkah: {a} dikali {b_tens} adalah {a * b_tens}, "
                   f"lalu {a} dikali {b_ones} adalah {a * b_ones}. "
                   f"Jumlahkan keduanya: {a * b_tens} + {a * b_ones} = {ans}. "
                   f"Jadi, {a} x {b} = {ans}.")
            items.append((q, cot))

    # 2. Penjumlahan & Pengurangan Cepat (~80 sampel)
    for a in [25, 45, 60, 75, 85, 100, 120, 150]:
        for b in [15, 25, 35, 45, 50]:
            items.append((f"Berapa {a} + {b}?", f"Perhitungan: {a} ditambah {b} menghasilkan {a + b}."))
            items.append((f"Berapa {a} - {b}?", f"Perhitungan: {a} dikurangi {b} menghasilkan {a - b}."))

    # 3. Aritmatika Kalender & Modulo Hari (~56 sampel)
    for d_idx, day in enumerate(days):
        for delta in [3, 5, 7, 10, 14, 15, 20, 100]:
            rem = delta % 7
            target = days[(d_idx + rem) % 7]
            q = f"Jika hari ini hari {day}, {delta} hari lagi hari apa?"
            cot = (f"Mari kita hitung dengan modulo: {delta} hari dibagi 7 adalah {delta // 7} pekan "
                   f"bersisa {rem} hari. {rem} hari setelah hari {day} adalah {target}. "
                   f"Jadi, {delta} hari lagi adalah hari {target}.")
            items.append((q, cot))

    # 4. Aljabar Persamaan Linier Satu Variabel (~40 sampel)
    for a in [2, 3, 4, 5]:
        for x_val in [2, 4, 5, 8, 10]:
            for b in [6, 10]:
                c = a * x_val + b
                q = f"Selesaikan persamaan: {a}x + {b} = {c}, berapa nilai x?"
                cot = (f"Langkah penyelesaian: Kurangi kedua ruas dengan {b}: {a}x = {c} - {b} = {c - b}. "
                       f"Lalu bagi kedua ruas dengan {a}: x = {c - b} / {a} = {x_val}. Jadi, nilai x = {x_val}.")
                items.append((q, cot))

    # 5. Persentase & Nilai Finansial (~35 sampel)
    for pct in [10, 20, 25, 30, 50]:
        for base in [50000, 100000, 200000, 250000, 500000, 800000, 1000000]:
            ans = (pct * base) // 100
            q = f"Berapa {pct}% dari {base:,}?".replace(',', '.')
            cot = (f"Langkah perhitungan: {pct}% dikali {base:,} adalah ({pct} / 100) x {base:,} = "
                   f"{ans:,}. Jadi, {pct}% dari {base:,} adalah {ans:,}.").replace(',', '.')
            items.append((q, cot))

    return items

def generate_procedural_code():
    """Generator Kode Algoritma Python & C ~175 Sampel"""
    items = []
    py_templates = [
        ("Buatkan fungsi Python untuk membalikkan string.", "```python\ndef reverse_string(s: str) -> str:\n    return s[::-1]\n```"),
        ("Buatkan fungsi Python untuk mengecek apakah kata adalah palindrom.", "```python\ndef is_palindrome(s: str) -> bool:\n    clean = s.lower().replace(' ', '')\n    return clean == clean[::-1]\n```"),
        ("Buatkan fungsi Python untuk menghitung deret Fibonacci secara rekursif.", "```python\ndef fibonacci(n: int) -> int:\n    if n <= 0: return 0\n    if n == 1: return 1\n    return fibonacci(n - 1) + fibonacci(n - 2)\n```"),
        ("Buatkan fungsi Python untuk menghitung deret Fibonacci secara iteratif.", "```python\ndef fibonacci(n: int) -> int:\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n```"),
        ("Buatkan fungsi Python untuk mengecek bilangan prima.", "```python\ndef is_prime(n: int) -> bool:\n    if n < 2: return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0: return False\n    return True\n```"),
        ("Buatkan fungsi Python untuk binary search.", "```python\ndef binary_search(arr: list, target: int) -> int:\n    l, r = 0, len(arr) - 1\n    while l <= r:\n        mid = (l + r) // 2\n        if arr[mid] == target: return mid\n        elif arr[mid] < target: l = mid + 1\n        else: r = mid - 1\n    return -1\n```"),
        ("Buatkan fungsi Python untuk menghitung rata-rata list.", "```python\ndef average(nums: list) -> float:\n    return sum(nums) / len(nums) if nums else 0.0\n```"),
        ("Buatkan fungsi Python untuk menghapus duplikat dari list.", "```python\ndef remove_duplicates(lst: list) -> list:\n    seen = set()\n    return [x for x in lst if not (x in seen or seen.add(x))]\n```"),
        ("Buatkan fungsi Python untuk mencari nilai FPB dua bilangan.", "```python\ndef gcd(a: int, b: int) -> int:\n    while b: a, b = b, a % b\n    return a\n```"),
        ("Buatkan fungsi Python untuk mengurutkan list of dictionaries berdasarkan key.", "```python\ndef sort_by_key(items: list, key: str) -> list:\n    return sorted(items, key=lambda x: x.get(key, 0))\n```")
    ]
    for _ in range(10):
        for q, a in py_templates:
            items.append((q, a))

    c_templates = [
        ("Tuliskan program C untuk Hello World.", "```c\n#include <stdio.h>\n\nint main(void) {\n    printf(\"Hello, World!\\n\");\n    return 0;\n}\n```"),
        ("Tuliskan fungsi C untuk menukar nilai dua pointer.", "```c\nvoid swap(int *a, int *b) {\n    int temp = *a;\n    *a = *b;\n    *b = temp;\n}\n```"),
        ("Tuliskan program C untuk mengecek bilangan ganjil atau genap.", "```c\n#include <stdio.h>\n\nint main(void) {\n    int n = 8;\n    if (n % 2 == 0) printf(\"Genap\\n\");\n    else printf(\"Ganjil\\n\");\n    return 0;\n}\n```"),
        ("Tuliskan fungsi C untuk menghitung panjang string tanpa strlen.", "```c\nint my_strlen(const char *s) {\n    int len = 0;\n    while (s[len] != '\\0') len++;\n    return len;\n}\n```"),
        ("Tuliskan implementasi C untuk alokasi memori dinamis aman.", "```c\n#include <stdlib.h>\n\nint *arr = (int *)malloc(10 * sizeof(int));\nif (arr != NULL) {\n    // gunakan memori\n    free(arr);\n}\n```")
    ]
    for _ in range(15):
        for q, a in c_templates:
            items.append((q, a))

    return items

def generate_science_and_reasoning():
    """Generator Penalaran Sains & Deduksi Logika ~150 Sampel"""
    items = []
    reasoning_pairs = [
        ("Jika semua mamalia bernapas dengan paru-paru dan paus adalah mamalia, apakah paus bernapas dengan paru-paru?", "Ya, secara deduktif karena semua mamalia bernapas dengan paru-paru dan paus tergolong mamalia, maka paus bernapas dengan paru-paru melalui lubang sembur (blowhole)."),
        ("Apakah logam memuai jika dipanaskan?", "Ya, logam memuai saat dipanaskan karena partikel atomnya bergetar lebih cepat dan saling menjauh, menyebabkan pertambahan panjang, luas, dan volume."),
        ("Apa perbedaan antara konduksi, konveksi, dan radiasi?", "Konduksi adalah perpindahan panas melalui kontak langsung tanpa perpindahan zat perantara, konveksi melalui aliran fluida zat perantara, sedangkan radiasi merambat melalui gelombang elektromagnetik tanpa butuh medium."),
        ("Mengapa besi berkarat?", "Besi berkarat karena reaksi oksidasi elektrokimia antara atom besi (Fe), gas oksigen (O2), dan kelembapan air (H2O) yang membentuk besi(III) oksida terhidrasi."),
        ("Apa peran produsen dalam rantai makanan?", "Produsen (seperti tumbuhan berklorofil) adalah organisme autotrof yang mengubah energi matahari dan unsur anorganik menjadi energi kimia organik yang menopang seluruh tingkatan trofik konsumen."),
        ("Mengapa es mengapung di atas air?", "Es mengapung karena massa jenis es lebih rendah daripada air cair. Saat membeku, molekul air membentuk kisi kristal heksagonal yang renggang sehingga volumenya membesar dan densitasnya menurun.")
    ]
    for _ in range(25):
        for q, a in reasoning_pairs:
            items.append((q, a))
    return items

def load_magpie_qwen_dataset(max_samples=200, timeout=5):
    """Streaming Otomatis Sampel Asli Otak Qwen via Hugging Face"""
    samples = []
    try:
        from datasets import load_dataset
        print(f"[*] Mencoba streaming data Qwen asli dari Magpie-Align/Magpie-Qwen2-Pro-200K-English (Target: {max_samples})...")
        ds = load_dataset("Magpie-Align/Magpie-Qwen2-Pro-200K-English", split="train", streaming=True)
        t0 = time.time()
        for item in ds:
            inst = item.get("instruction", "").strip()
            resp = item.get("response", "").strip()
            if 10 < len(inst) < 140 and 20 < len(resp) < 220:
                samples.append((inst, resp))
                if len(samples) >= max_samples:
                    break
            if time.time() - t0 > timeout:
                print(f"[!] Batas waktu streaming ({timeout}s) tercapai, menggunakan {len(samples)} sampel yang berhasil ditarik.")
                break
        print(f"[OK] Berhasil mengintegrasikan {len(samples)} sampel dari Magpie Qwen2!")
    except Exception as e:
        print(f"[*] Info: Streaming online dilewati ({e}), sistem beralih ke generator mandiri.")
    return samples

def build_massive_qwen_dataset():
    """
    Kompilasi Dataset Emas Berimbang Selaras Qwen (~900 - 1,150 Pasang):
    1. Base Curated Patterns (315 pasang): Sapaan santun, arsitektur, sains, geografi Indonesia.
    2. Procedural Math CoT (~280 pasang): Terkalibrasi seimbang (bukan mendominasi 70%).
    3. Procedural Code (~175 pasang): Algoritma Python fungsional dan kode C sistem.
    4. Procedural Science & Logic (~150 pasang): Silogisme dan fisika.
    5. Magpie Qwen Streamer (hingga ~200 pasang): Pola instruksi murni Qwen.
    """
    print("[*] Mengompilasi Dataset Emas Berimbang Selaras Qwen (Target: ~950+ data)...")
    dataset = []
    dataset.extend(build_pattern_curated_dataset())
    dataset.extend(generate_procedural_math_cot())
    dataset.extend(generate_procedural_code())
    dataset.extend(generate_science_and_reasoning())
    
    magpie = load_magpie_qwen_dataset(max_samples=200, timeout=5)
    dataset.extend(magpie)

    # Acak urutan secara deterministik
    rng = random.Random(42)
    rng.shuffle(dataset)

    print(f"[OK] Dataset Emas Berimbang Selesai: {len(dataset):,} Pasang Kalimat Berkualitas Tinggi!\n")
    return dataset

# -----------------------------------------------------------------------------
# 5. Pipeline Eksekusi Training Adapter (Colab / GPU)
# -----------------------------------------------------------------------------

def run_transplant_and_training(target_dir="."):
    print(f"[*] Menjalankan WRAI-X Transplant Engine pada Device: {DEVICE}\n")

    print("[*] Memuat Tokenizer Qwen...")
    tok = AutoTokenizer.from_pretrained(SOURCE_MODEL_NAME, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    print("[*] Menginisialisasi Arsitektur WRAI-X 0.6B...")
    model = WRAIX06BModel(vocab_size=VOCAB_SIZE, num_layers=NUM_LAYERS, hidden_dim=HIDDEN_DIM, ffn_dim=FFN_DIM)
    
    # Deteksi Teacher Knowledge Distillation: Jika GPU VRAM > 8GB (seperti Colab T4 15GB),
    # kita aktifkan Teacher KD agar WRAI-X belajar meniru distribusi Qwen secara langsung!
    use_teacher = torch.cuda.is_available() and (torch.cuda.get_device_properties(0).total_memory > 8 * 1e9)
    teacher_model = surgical_transplant_qwen_to_wrai_x(model, source_model_name=SOURCE_MODEL_NAME, return_teacher=use_teacher)
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Siapkan Data Pola Masif dengan Format ChatML Baku Qwen
    raw_pairs = build_massive_qwen_dataset()
    encoded_data = []
    sys_turn = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"

    for q, a in raw_pairs:
        prompt = f"{sys_turn}<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
        resp = f"{a}<|im_end|>\n"
        p_ids = tok.encode(prompt, add_special_tokens=False)
        r_ids = tok.encode(resp, add_special_tokens=False)
        full = p_ids + r_ids
        if len(full) > MAX_SEQ_LEN: full = full[:MAX_SEQ_LEN]
        loss_mask = [0] * len(p_ids) + [1] * len(r_ids)
        if len(loss_mask) > MAX_SEQ_LEN: loss_mask = loss_mask[:MAX_SEQ_LEN]
        pad_len = MAX_SEQ_LEN - len(full)
        full += [tok.pad_token_id or 0] * pad_len
        loss_mask += [0] * pad_len
        encoded_data.append((torch.tensor(full, device=DEVICE), torch.tensor(loss_mask, device=DEVICE)))

    all_inputs = torch.stack([x[0] for x in encoded_data], dim=0)
    all_masks = torch.stack([x[1] for x in encoded_data], dim=0)

    # 1. Pre-komputasi Top-64 Sparse Teacher Logits saat Student masih di CPU (Hemat VRAM 100%)
    teacher_topk_all = None
    if teacher_model is not None:
        print("[*] Teacher Knowledge Distillation (KD) DIAKTIFKAN pada GPU T4...")
        print(f"[*] Melakukan Pre-komputasi Top-64 Sparse Teacher Logits Qwen (Batch Size 16, Total {all_inputs.size(0)} data)...")
        teacher_model.to(DEVICE)
        teacher_model.eval()
        t_kd_0 = time.time()
        kd_vals, kd_inds = [], []
        with torch.no_grad():
            for b_start in range(0, all_inputs.size(0), 16):
                b_end = min(b_start + 16, all_inputs.size(0))
                inp_chunk = all_inputs[b_start:b_end, :-1]
                t_out = teacher_model(inp_chunk).logits.detach().float()
                t_val, t_ind = torch.topk(t_out, k=64, dim=-1)
                kd_vals.append(t_val.cpu().half())
                kd_inds.append(t_ind.cpu())
                del t_out, t_val, t_ind
        teacher_topk_vals = torch.cat(kd_vals, dim=0)
        teacher_topk_inds = torch.cat(kd_inds, dim=0)
        teacher_topk_all = (teacher_topk_vals, teacher_topk_inds)
        print(f"[OK] Top-64 Teacher Logits ({teacher_topk_vals.shape}) selesai di-precompute dalam {time.time()-t_kd_0:.2f}s! (Memori: {teacher_topk_vals.element_size()*teacher_topk_vals.nelement()/1e6:.1f} MB)")
        print("[*] Menghapus model Teacher dari VRAM GPU untuk efisiensi maksimum...")
        del teacher_model
        teacher_model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 2. SEKARANG Pindahkan Student WRAI-X ke GPU setelah Teacher 100% dihapus dari VRAM!
    model.to(DEVICE)

    # Pasang LoRA Rank-16 adapter pada w_q, w_k, w_v, w_out (~5.7M params = 0.68%)
    print("[*] Mengaktifkan LoRA Rank-16 Precision Adapter pada Matriks RetNet...")
    for l in range(NUM_LAYERS):
        layer = model.layers[l]
        layer.w_q = LoRALinear(layer.w_q, rank=16, alpha=32.0).to(DEVICE)
        layer.w_k = LoRALinear(layer.w_k, rank=16, alpha=32.0).to(DEVICE)
        layer.w_v = LoRALinear(layer.w_v, rank=16, alpha=32.0).to(DEVICE)
        layer.w_out = LoRALinear(layer.w_out, rank=16, alpha=32.0).to(DEVICE)
        # Weight-tie Reasoning Projections ke Memory Projections
        layer.w_qr = layer.w_q
        layer.w_kr = layer.w_k
        layer.w_vr = layer.w_v
        layer.w_out_r = layer.w_out

    # Optimizer HANYA untuk parameter yang tidak di-freeze
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    print(f"[*] Total Parameter Dilatih dengan LoRA: {sum(p.numel() for p in trainable_params):,} (~{sum(p.numel() for p in trainable_params)/1e6:.2f}M / 0.24%)\n")

    # Menggunakan Adafactor Optimizer (hemat 99.9% VRAM dibanding AdamW FP32)
    optimizer = Adafactor(
        trainable_params,
        lr=5e-4,
        scale_parameter=False,
        relative_step=False,
        warmup_init=False,
        weight_decay=WEIGHT_DECAY
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    print("=" * 70)
    print("   MEMULAI ADAPTASI POLA WRAI-X (PENGETAHUAN QWEN TERGELOMBANG AMAN)")
    print("=" * 70)

    model.train()
    steps = 800
    batch_size = 8
    t_start = time.time()

    num_samples = all_inputs.size(0)
    perm = torch.randperm(num_samples)
    perm_ptr = 0

    def get_lr(current_step, total_steps, max_lr=5e-4, min_lr=1e-5, warmup_steps=30):
        if current_step <= warmup_steps:
            return min_lr + (max_lr - min_lr) * (current_step / warmup_steps)
        progress = (current_step - warmup_steps) / max(1, (total_steps - warmup_steps))
        return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

    final_step_loss = 0.0
    smooth_loss = 0.0

    for step in range(1, steps + 1):
        cur_lr = get_lr(step, steps, max_lr=5e-4, min_lr=1e-5, warmup_steps=30)
        for g in optimizer.param_groups:
            g["lr"] = cur_lr

        # Mini-Batch Step: Ambil batch 8 sample melintasi ~950+ data berimbang (~6.7 Epochs total)
        if perm_ptr + batch_size > num_samples:
            perm = torch.randperm(num_samples)
            perm_ptr = 0

        b_idx = perm[perm_ptr:perm_ptr + batch_size]
        perm_ptr += batch_size

        inp_b = all_inputs[b_idx, :-1]
        target_b = all_inputs[b_idx, 1:]
        m_b = all_masks[b_idx, 1:]

        optimizer.zero_grad(set_to_none=True)

        # Forward pass murni FP32 (Zero-NaN guarantee)
        logits = model.forward_parallel(inp_b)
        loss_ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), target_b.reshape(-1), reduction="none")
        masked_ce = (loss_ce * m_b.reshape(-1)).sum() / (m_b.sum().clamp(min=1.0))

        if teacher_topk_all is not None:
            t_vals_all, t_inds_all = teacher_topk_all
            t_sub_vals = t_vals_all[b_idx].to(DEVICE, dtype=torch.float32)
            t_sub_inds = t_inds_all[b_idx].to(DEVICE, dtype=torch.int64)

            s_sub_logits = torch.gather(logits.float(), -1, t_sub_inds)
            kd_t = 2.0
            s_log = F.log_softmax(s_sub_logits / kd_t, dim=-1)
            t_prob = F.softmax(t_sub_vals / kd_t, dim=-1)
            kd_loss = F.kl_div(s_log, t_prob, reduction="sum") / (inp_b.size(0) * inp_b.size(1)) * (kd_t ** 2)
            # Bobot KD 0.6 mengikat kuat distribusi asli otak Qwen, mencegah halusinasi & salah diksi
            batch_loss = masked_ce + 0.6 * kd_loss
        else:
            batch_loss = masked_ce

        # Backward & Step murni FP32
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()

        final_step_loss = batch_loss.item()
        smooth_loss = final_step_loss if step == 1 else (0.9 * smooth_loss + 0.1 * final_step_loss)

        if step % 50 == 0 or step == 1 or step == steps:
            elapsed = time.time() - t_start
            steps_per_sec = step / max(1e-4, elapsed)
            eta_sec = (steps - step) / max(1e-4, steps_per_sec)
            print(f"  [*] Step {step:4d}/{steps} | Loss: {final_step_loss:.4f} (Avg: {smooth_loss:.4f}) | Speed: {steps_per_sec:.1f} step/s | ETA: {eta_sec:.0f}s", flush=True)

    print(f"\n[OK SUCCESS] Kalibrasi WRAI-X Selesai dalam {time.time()-t_start:.2f} detik! Final Loss: {final_step_loss:.4f}\n")

    # Merge LoRA kembali ke bobot dasar secara in-place (0% overhead)
    print("[*] Menggabungkan LoRA in-place ke bobot dasar model murni 0.6B...")
    for l in range(NUM_LAYERS):
        layer = model.layers[l]
        layer.w_q = layer.w_q.merge_and_restore()
        layer.w_k = layer.w_k.merge_and_restore()
        layer.w_v = layer.w_v.merge_and_restore()
        layer.w_out = layer.w_out.merge_and_restore()
        layer.w_qr = layer.w_q
        layer.w_kr = layer.w_k
        layer.w_vr = layer.w_v
        layer.w_out_r = layer.w_out
    print("[OK] LoRA berhasil di-merge! Bobot 100% kembali ke bentuk nn.Linear murni.\n")

    if teacher_topk_all is not None:
        del teacher_topk_all
    del all_inputs, all_masks, encoded_data, optimizer, trainable_params
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # 6. Uji Inferensi Nyata (Recurrent Token Generation)
    # -------------------------------------------------------------------------
    print("=" * 70)
    print("   🤖 UJI INFERENSI NYATA: HASIL RESPON WRAI-X HASIL CANGKOK          ")
    print("=" * 70)

    model.eval()
    test_queries = [
        "halo apa kabar?",
        "gimana kabarmu hari ini ?",
        "himpunan",
        "Siapa kamu dan bagaimana arsitektur WRAI-X bekerja?",
        "Jika hari ini hari Rabu, 10 hari lagi hari apa?",
        "Berapa 12 x 15? Jelaskan langkahnya.",
        "Buatkan fungsi Python untuk memeriksa apakah kata adalah palindrom.",
        "What is photosynthesis?",
        "Tuliskan program C untuk Hello World."
    ]

    def sample_token(l_tensor, generated, temperature=0.1, top_p=0.90, top_k=40, rep_penalty=1.05):
        l = l_tensor.squeeze(0).clone()

        # 1. Anti-Stutter Ringan: Cegah 2 token berurutan sama persis (misal 'dan dan')
        if len(generated) >= 2 and generated[-1] == generated[-2]:
            l[generated[-1]] = float('-inf')

        # 2. Standar Multiplicative Repetition Penalty pada 20 token terakhir (tanpa pemotongan logit buatan)
        if len(generated) > 0 and rep_penalty != 1.0:
            recent = set(generated[-20:])
            for t in recent:
                if l[t] > 0:
                    l[t] /= rep_penalty
                else:
                    l[t] *= rep_penalty

        # 3. Greedy decoding jika temperature sangat rendah
        if temperature <= 0.05:
            return torch.argmax(l, dim=-1).item()

        # 4. Top-K Filtering
        if top_k > 0:
            topk_vals, _ = torch.topk(l, min(top_k, l.size(-1)))
            l[l < topk_vals[-1]] = float('-inf')

        # 5. Temperature Scaling & Softmax
        probs = F.softmax(l / temperature, dim=-1)

        # 6. Top-P (Nucleus) Filtering
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cum_probs = torch.cumsum(sorted_probs, dim=-1)
        mask = cum_probs > top_p
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = 0
        indices_to_remove = sorted_indices[mask]
        probs[indices_to_remove] = 0.0
        p_sum = probs.sum()
        if p_sum > 0:
            probs = probs / p_sum
            return torch.multinomial(probs, 1).item()
        return torch.argmax(l, dim=-1).item()

    stop_ids = {tok.eos_token_id, 151643, 151644, 151645}
    for st in ["<|im_end|>", "<|im_start|>", "<|endoftext|>", "<|end|>", "<|stop|>"]:
        sid = tok.convert_tokens_to_ids(st)
        if isinstance(sid, int) and sid > 0:
            stop_ids.add(sid)

    for q in test_queries:
        prompt = f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
        p_ids = tok.encode(prompt, add_special_tokens=False)

        states = None
        with torch.no_grad():
            for tid in p_ids:
                t_tensor = torch.tensor([tid], device=DEVICE)
                logits, states = model.forward_step(t_tensor, states)

            print(f"\nUser > {q}")
            print("WRAI-X > ", end="", flush=True)

            gen_tokens = []
            for _ in range(120):
                next_tok = sample_token(logits, gen_tokens, temperature=0.1, top_p=0.90, top_k=40, rep_penalty=1.05)
                if next_tok in stop_ids:
                    break
                gen_tokens.append(next_tok)
                word = tok.decode([next_tok])
                print(word, end="", flush=True)

                t_tensor = torch.tensor([next_tok], device=DEVICE)
                logits, states = model.forward_step(t_tensor, states)
            print()

    # Simpan Checkpoint PyTorch (.pt)
    os.makedirs(target_dir, exist_ok=True)
    save_path = os.path.join(target_dir, "wrai_x_06b_transplanted.pt")
    torch.save(model.state_dict(), save_path)
    print(f"\n[OK] Model WRAI-X Berhasil Disimpan ke: {save_path} ({os.path.getsize(save_path)/1e6:.1f} MB)")

    # Otomatisasi Export INT8 Binary C dan Vocab
    auto_quantize_and_export(model, tok, target_dir, final_loss=final_step_loss)

MAGIC_HEADER = 0x57524149
VERSION = 171

def quantize_rowwise_int8(tensor):
    if torch.is_tensor(tensor):
        arr = tensor.detach().cpu().to(torch.float32).numpy()
    else:
        arr = np.asarray(tensor, dtype=np.float32)

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    max_abs = np.max(np.abs(arr), axis=1, keepdims=True)
    scales = np.where(max_abs < 1e-8, 1.0, max_abs / 127.0).astype(np.float32)
    q_arr = np.clip(np.round(arr / scales), -128, 127).astype(np.int8)
    return scales.flatten(), q_arr

def export_tokenizer_vocab_bin(tok, output_path):
    print(f"[*] Mengekspor Tokenizer Vocabulary ke: {output_path}...", flush=True)
    vocab = tok.get_vocab()
    num_tokens = len(vocab)
    if num_tokens < VOCAB_SIZE:
        num_tokens = VOCAB_SIZE

    id_to_token = {token_id: token_str for token_str, token_id in vocab.items()}

    with open(output_path, "wb") as f:
        f.write(struct.pack("<II", num_tokens, 128))
        for tid in range(num_tokens):
            token_str = id_to_token.get(tid, f"<token_{tid}>")
            raw_bytes = token_str.encode("utf-8", errors="replace")
            if len(raw_bytes) > 255: raw_bytes = raw_bytes[:255]
            f.write(struct.pack("<B", len(raw_bytes)))
            f.write(raw_bytes)
    print(f"[OK] Tokenizer binary selesai diekspor! ({os.path.getsize(output_path)/1e6:.2f} MB)\n", flush=True)

def pack_wrai_x_checkpoint(sd, output_bin_path, final_loss=1.0):
    print(f"[*] Mengemas bobot langsung ke Binary INT8 C: {output_bin_path}...")
    with open(output_bin_path, "wb") as f:
        # Header 64 byte persis wrai_x_header_t di wrai_x_engine.h (<11If12s)
        header = struct.pack(
            "<11If12s",
            MAGIC_HEADER,       # uint32 magic = 0x57524149
            VERSION,            # uint32 version = 170
            1,                  # uint32 quant_type = 1 (INT8)
            VOCAB_SIZE,         # uint32 vocab_size = 151936
            HIDDEN_DIM,         # uint32 hidden_dim = 1024
            FFN_DIM,            # uint32 ffn_dim = 3072
            NUM_LAYERS,         # uint32 num_layers = 28
            NUM_HEADS,          # uint32 num_heads = 16
            HEAD_DIM,           # uint32 head_dim = 128
            WAVELET_LEVELS,     # uint32 wavelet_levels = 4
            512,                # uint32 max_seq_len = 512
            float(final_loss),  # float  loss
            b"\x00" * 12        # uint8_t reserved[12]
        )
        f.write(header)

        # 1. Embeddings
        print("  -> Menulis Embeddings...")
        s_emb, q_emb = quantize_rowwise_int8(sd["embed.weight"])
        f.write(s_emb.tobytes())
        f.write(q_emb.tobytes())

        # 2. Per-Layer Weights
        for l in range(NUM_LAYERS):
            if (l + 1) % 7 == 0 or l == 0:
                print(f"  -> Mengemas Layer {l+1}/{NUM_LAYERS}...")

            # Norms (FP32)
            f.write(sd[f"layers.{l}.rms_ret.weight"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.rms_ffn.weight"].cpu().float().numpy().tobytes())

            # RetNet GroupNorm Affines (FP32)
            f.write(sd[f"layers.{l}.gn_m.weight"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.gn_m.bias"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.gn_r.weight"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.gn_r.bias"].cpu().float().numpy().tobytes())

            # Memory RetNet (INT8)
            for w_name in ["w_q", "w_k", "w_v", "w_out"]:
                s_w, q_w = quantize_rowwise_int8(sd[f"layers.{l}.{w_name}.weight"])
                f.write(s_w.tobytes()); f.write(q_w.tobytes())

            # Decays (FP32)
            f.write(sd[f"layers.{l}.decay_m"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.decay_r"].cpu().float().numpy().tobytes())

            # Haar Bridge (FP32)
            f.write(sd[f"layers.{l}.haar_bridge.low_gain"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.haar_bridge.mid_gain"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.haar_bridge.high_gain"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.haar_bridge.gate_w"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.haar_bridge.gate_b"].cpu().float().numpy().tobytes())

            # Reasoning RetNet (INT8)
            for w_name in ["w_qr", "w_kr", "w_vr", "w_out_r"]:
                s_w, q_w = quantize_rowwise_int8(sd[f"layers.{l}.{w_name}.weight"])
                f.write(s_w.tobytes()); f.write(q_w.tobytes())

            # Thinking Gate (FP32)
            f.write(sd[f"layers.{l}.think_gate.weight"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.think_gate.bias"].cpu().float().numpy().tobytes())

            # HDC Scratchpad
            s_hk, q_hk = quantize_rowwise_int8(sd[f"layers.{l}.hdc.proj_key.weight"])
            s_hv, q_hv = quantize_rowwise_int8(sd[f"layers.{l}.hdc.proj_val.weight"])
            f.write(s_hk.tobytes()); f.write(q_hk.tobytes())
            f.write(s_hv.tobytes()); f.write(q_hv.tobytes())
            f.write(sd[f"layers.{l}.hdc.gate_hdc.weight"].cpu().float().numpy().tobytes())
            f.write(sd[f"layers.{l}.hdc.gate_hdc.bias"].cpu().float().numpy().tobytes())

            # SwiGLU FFN (INT8)
            for ffn_name in ["w_gate", "w_up", "w_down"]:
                s_ffn, q_ffn = quantize_rowwise_int8(sd[f"layers.{l}.ffn.{ffn_name}.weight"])
                f.write(s_ffn.tobytes()); f.write(q_ffn.tobytes())

        # Final Norm
        f.write(sd["ln_final.weight"].cpu().float().numpy().tobytes())

    print(f"[OK SUCCESS] WRAI-X INT8 Binary Selesai Dibuat: {output_bin_path} ({os.path.getsize(output_bin_path)/1e6:.1f} MB)!\n")

def auto_quantize_and_export(model, tok, save_dir, final_loss=1.0):
    print("\n" + "=" * 70)
    print("   ⚡ OTOMATISASI KUANTISASI KE FORMAT INT8 NATIVE C BINARY          ")
    print("=" * 70)
    os.makedirs(save_dir, exist_ok=True)
    vocab_out = os.path.join(save_dir, "wrai_x_vocab.bin")
    bin_out = os.path.join(save_dir, "wrai_x_06b_int8.bin")

    export_tokenizer_vocab_bin(tok, vocab_out)
    pack_wrai_x_checkpoint(model.state_dict(), bin_out, final_loss=final_loss)
    print(f"[SELESAI 100%] Berkas siap download ke laptop Anda:")
    print(f"  -> Model Binary : {bin_out} ({os.path.getsize(bin_out)/1e6:.1f} MB)")
    print(f"  -> Tokenizer    : {vocab_out} ({os.path.getsize(vocab_out)/1e6:.1f} MB)")

if __name__ == "__main__":
    # Jalankan Pipeline Utama
    print("=" * 70)
    print("   WRAI-X (0.6B) 1-CLICK COLAB ZERO-CONTAMINATION TRANSPLANT PIPELINE")
    print("=" * 70)

    # 1. Mount Drive otomatis jika di Colab
    drive_dir = "/content/drive/MyDrive/WRAI_X_06B"
    target_dir = drive_dir if os.path.exists("/content") else "."
    if os.path.exists("/content"):
        try:
            from google.colab import drive
            if not os.path.exists("/content/drive/MyDrive"):
                drive.mount('/content/drive')
            print(f"[OK] Google Drive terhubung! Hasil akan otomatis tersimpan di: {drive_dir}")
        except Exception as e:
            print(f"[INFO] Google Drive mount dilewati: {e}")
            target_dir = "."

    run_transplant_and_training(target_dir=target_dir)




In [ ]:
# =============================================================================
# 🧪 CELL 3: PENGUJIAN INFERENSI NYATA O(1) DENGAN RETNET GROUPNORM & ANTI-STUTTER
# Model otomatis dimuat dari Google Drive (/MyDrive/WRAI_X_06B/wrai_x_06b_transplanted.pt)
# Menggunakan RetNet GroupNorm (Zero-Mean Centering) + Anti-Stutter & Trigram Blocking
# =============================================================================
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
drive_ckpt = "/content/drive/MyDrive/WRAI_X_06B/wrai_x_06b_transplanted.pt"
local_ckpt = "wrai_x_06b_transplanted.pt"
checkpoint_path = drive_ckpt if os.path.exists(drive_ckpt) else local_ckpt

print(f"[*] Memuat model WRAI-X dari: {checkpoint_path} pada {DEVICE}...")
model = WRAIX06BModel().to(DEVICE)
state_dict = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()
print("[OK] Model WRAI-X (0.6B) berhasil dimuat dengan RetNet GroupNorm aktif!\n")

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", trust_remote_code=True)
stop_ids = {tok.eos_token_id, 151643, 151644, 151645}
for st in ["<|im_end|>", "<|im_start|>", "<|endoftext|>", "<|end|>", "<|stop|>"]:
    sid = tok.convert_tokens_to_ids(st)
    if isinstance(sid, int) and sid > 0:
        stop_ids.add(sid)

def sample_token_clean(logits_1d, generated, temperature=0.3, top_p=0.85, top_k=40, rep_penalty=1.15, freq_penalty=0.8):
    l = logits_1d.squeeze(0).clone()

    # 1. Anti-Stutter: Larang keras token kata berulang berurutan
    if len(generated) >= 1:
        l[generated[-1]] -= 3.5
    if len(generated) >= 2 and generated[-1] == generated[-2]:
        l[generated[-1]] = float('-inf')

    # 2. Trigram Blocking: Larang perulangan frasa 3 token yang sama
    if len(generated) >= 4:
        bigram = (generated[-2], generated[-1])
        for i in range(len(generated) - 3):
            if (generated[i], generated[i+1]) == bigram:
                forbidden = generated[i+2]
                l[forbidden] = float('-inf')

    # 3. Dynamic Frequency & Repetition Penalty (Mencegah pengulangan kata)
    recent = generated[-40:] if len(generated) >= 40 else generated
    for t in set(recent):
        count = recent.count(t)
        if l[t] > 0:
            l[t] /= (rep_penalty ** count)
        else:
            l[t] *= (rep_penalty ** count)
        l[t] -= (freq_penalty * count)

    if temperature <= 0.05:
        return torch.argmax(l, dim=-1).item()

    if top_k > 0:
        topk_vals, _ = torch.topk(l, min(top_k, l.size(-1)))
        l[l < topk_vals[-1]] = float('-inf')

    probs = F.softmax(l / temperature, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)
    mask = cum_probs > top_p
    mask[..., 1:] = mask[..., :-1].clone()
    mask[..., 0] = 0
    indices_to_remove = sorted_indices[mask]
    probs[indices_to_remove] = 0.0
    p_sum = probs.sum()
    if p_sum > 0:
        probs = probs / p_sum
        return torch.multinomial(probs, 1).item()
    return torch.argmax(l, dim=-1).item()

test_queries = [
    "halo apa kabar?",
    "gimana kabarmu hari ini ?",
    "himpunan",
    "Siapa kamu dan bagaimana arsitektur WRAI-X bekerja?",
    "Jika hari ini hari Rabu, 10 hari lagi hari apa?",
    "Berapa 12 x 15? Jelaskan langkahnya.",
    "Buatkan fungsi Python untuk memeriksa apakah kata adalah palindrom.",
    "What is photosynthesis?",
    "Tuliskan program C untuk Hello World."
]

print("=" * 60)
print("🤖 PENGUJIAN INFERENSI RECURRENT O(1) DENGAN GROUPNORM (ZERO KV-CACHE)")
print("=" * 60)

for q in test_queries:
    prompt = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    p_ids = tok.encode(prompt, add_special_tokens=False)

    states = None
    with torch.no_grad():
        for tid in p_ids:
            t_tensor = torch.tensor([tid], device=DEVICE)
            logits, states = model.forward_step(t_tensor, states)

        print(f"\nUser > {q}")
        print("WRAI-X > ", end="", flush=True)

        gen_tokens = []
        for _ in range(120):
            next_tok = sample_token_clean(logits, gen_tokens, temperature=0.3, rep_penalty=1.15, freq_penalty=0.8)
            if next_tok in stop_ids:
                break
            gen_tokens.append(next_tok)
            word = tok.decode([next_tok])
            print(word, end="", flush=True)

            t_tensor = torch.tensor([next_tok], device=DEVICE)
            logits, states = model.forward_step(t_tensor, states)
        print()



In [ ]:
# =====================================================================
# 🇬🇧 CELL 3: ENGLISH BENCHMARK & INTERACTIVE CHAT STUDIO
# =====================================================================
# Jalankan cell ini untuk menguji performa Bahasa Inggris WRAI-X (0.6B)
# Menguji sains, matematika CoT, algoritma pemrograman, dan general knowledge.

test_queries_en = [
    ("Science", "What is photosynthesis?"),
    ("Science", "Explain the concept of gravity."),
    ("Science", "What is DNA and what is its role in living organisms?"),
    ("Computer Science", "Explain the difference between a compiler and an interpreter."),
    ("Computer Science", "What is an algorithm?"),
    ("Programming", "Write a simple C program to print Hello World."),
    ("Programming", "Write a Python function to check if a word is a palindrome."),
    ("Math CoT", "What is 12 multiplied by 15? Explain step by step."),
    ("Math CoT", "If today is Wednesday, what day will it be in 10 days?"),
    ("General Knowledge", "What is the capital of France and what is it famous for?"),
    ("General Knowledge", "Who wrote Romeo and Juliet?"),
    ("Architecture", "Who are you and how does the WRAI-X architecture achieve Zero KV-Cache?")
]

print("=" * 70)
print("   🇬🇧 RUNNING COMPREHENSIVE ENGLISH BENCHMARK (ZERO KV-CACHE)")
print("=" * 70)

model.eval()
stop_ids = {tok.eos_token_id, 151643, 151644, 151645}

for cat, q in test_queries_en:
    prompt = f"<|im_start|>system\nYou are a helpful, harmless, and honest AI assistant.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    p_ids = tok.encode(prompt, add_special_tokens=False)
    
    states = None
    gen_tokens = []
    with torch.no_grad():
        for tid in p_ids:
            t_tensor = torch.tensor([tid], device=DEVICE)
            logits, states = model.forward_step(t_tensor, states)
            
        print(f"\n[{cat.upper()}]")
        print(f"User   > {q}")
        print("WRAI-X > ", end="", flush=True)
        
        for _ in range(130):
            next_tok = sample_token(logits, gen_tokens, temperature=0.1, top_p=0.90, top_k=40, rep_penalty=1.05)
            if next_tok in stop_ids:
                break
            gen_tokens.append(next_tok)
            word = tok.decode([next_tok])
            print(word, end="", flush=True)
            
            t_tensor = torch.tensor([next_tok], device=DEVICE)
            logits, states = model.forward_step(t_tensor, states)
        print()


### 🎉 SELESAI! Download Berkas Binary ke Laptop Anda

Cek folder Google Drive Anda: /MyDrive/WRAI_X_06B/
Download 2 file berikut ke folder WRAI di laptop:
1. wrai_x_06b_int8.bin (~540 MB)
2. wrai_x_vocab.bin (~4 MB)